In [ ]:
import os
import gc
import re
import json
import math
import time
import random
import warnings
import unicodedata
import html as html_lib
from pathlib import Path
from dataclasses import dataclass, field
from typing import Optional, List, Dict, Tuple, Any

import numpy as np
import pandas as pd

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torch.amp import autocast, GradScaler
from torch.optim import AdamW

from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import (
    accuracy_score,
    precision_recall_fscore_support,
    f1_score,
    confusion_matrix,
    classification_report,
    roc_auc_score,
)
from sklearn.preprocessing import LabelEncoder
from sklearn.utils.class_weight import compute_class_weight

from transformers import (
    AutoTokenizer,
    AutoModel,
    AutoConfig,
    get_linear_schedule_with_warmup,
    get_cosine_schedule_with_warmup,
)

import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import seaborn as sns

warnings.filterwarnings("ignore")

try:
    import optuna
    OPTUNA_AVAILABLE = True
except ImportError:
    OPTUNA_AVAILABLE = False

print("Imports loaded successfully")
print(f"PyTorch version: {torch.__version__}")
print(f"Optuna available: {OPTUNA_AVAILABLE}")

Imports loaded successfully
PyTorch version: 2.5.1+cu118
Optuna available: True


In [ ]:
def detect_device():
    if torch.cuda.is_available():
        device = torch.device("cuda")
        props = torch.cuda.get_device_properties(0)
        total_vram_gb = props.total_memory / (1024 ** 3)
        print(f"CUDA device detected: {props.name}")
        print(f"Total VRAM: {total_vram_gb:.2f} GB")
        torch.backends.cudnn.benchmark = True
        torch.backends.cuda.matmul.allow_tf32 = True
        torch.backends.cudnn.allow_tf32 = True
    else:
        device = torch.device("cpu")
        print("CUDA not available. Falling back to CPU.")
    return device


DEVICE = detect_device()
USE_AMP = DEVICE.type == "cuda"
NUM_WORKERS = 0 if os.name == "nt" else 4
PIN_MEMORY = DEVICE.type == "cuda"

print(f"Using device: {DEVICE}")
print(f"Mixed precision enabled: {USE_AMP}")
print(f"Num workers: {NUM_WORKERS}")


def clear_memory():
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
        torch.cuda.synchronize()


def print_gpu_memory(tag=""):
    if torch.cuda.is_available():
        allocated = torch.cuda.memory_allocated() / (1024 ** 3)
        reserved = torch.cuda.memory_reserved() / (1024 ** 3)
        print(f"[{tag}] GPU allocated: {allocated:.2f} GB | reserved: {reserved:.2f} GB")


clear_memory()
print_gpu_memory("initial")

CUDA device detected: Tesla T4
Total VRAM: 14.84 GB
Using device: cuda
Mixed precision enabled: True
Num workers: 0
[initial] GPU allocated: 0.00 GB | reserved: 0.00 GB


In [ ]:
def set_seed(seed: int = 42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    os.environ["PYTHONHASHSEED"] = str(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False


GLOBAL_SEED = 42
set_seed(GLOBAL_SEED)
print(f"Global seed set to {GLOBAL_SEED}")

Global seed set to 42


In [ ]:
@dataclass
class PathConfig:
    root: Path = Path(r"C:/Users/VGST/Desktop//FIRE-2026/DravidianCodeMix 2026")

    task1_train: Path = field(init=False)
    task1_val: Path = field(init=False)
    task1_test: Path = field(init=False)
    task1_test_labeled: Path = field(init=False)

    task2_train: Path = field(init=False)
    task2_val: Path = field(init=False)
    task2_test: Path = field(init=False)
    task2_test_labeled: Path = field(init=False)

    task3_train: Path = field(init=False)
    task3_val: Path = field(init=False)
    task3_test: Path = field(init=False)
    task3_test_labeled: Path = field(init=False)

    output_root: Path = field(init=False)

    def __post_init__(self):
        self.task1_train = self.root / "SUBTASK 1" / "train.csv"
        self.task1_val = self.root / "SUBTASK 1" / "validation.csv"
        self.task1_test = self.root / "SUBTASK 1" / "test.csv"
        self.task1_test_labeled = self.root / "SUBTASK 1" / "test_labeled.csv"

        self.task2_train = self.root / "SUBTASK 2" / "train_multitask.csv"
        self.task2_val = self.root / "SUBTASK 2" / "dev_multitask.csv"
        self.task2_test = self.root / "SUBTASK 2" / "multitask_test_withoutlabel.csv"
        self.task2_test_labeled = self.root / "SUBTASK 2" / "test with label.csv"

        self.task3_train = self.root / "SUBTASK 3" / "Kannada_train.csv"
        self.task3_val = self.root / "SUBTASK 3" / "Kannada_val.csv"
        self.task3_test = self.root / "SUBTASK 3" / "Kannada_test_without_labels.csv"
        self.task3_test_labeled = self.root / "SUBTASK 3" / "Kannada_test.csv"

        self.output_root = self.root / "outputs"
        for sub in ["checkpoints", "logs", "predictions", "plots", "submissions"]:
            (self.output_root / sub).mkdir(parents=True, exist_ok=True)


@dataclass
class TrainConfig:
    max_seq_len: int = 192
    batch_size: int = 8
    eval_batch_size: int = 16
    grad_accum_steps: int = 4
    epochs: int = 5
    lr: float = 2e-5
    weight_decay: float = 0.01
    warmup_ratio: float = 0.1
    dropout: float = 0.1
    label_smoothing: float = 0.0
    max_grad_norm: float = 1.0
    early_stopping_patience: int = 2
    scheduler_type: str = "linear"
    loss_type: str = "weighted_ce"
    focal_gamma: float = 2.0
    seed: int = 42


MODEL_REGISTRY = {
    "xlm-roberta-base": "xlm-roberta-base",
    "muril-base": "google/muril-base-cased",
    "mbert-base": "bert-base-multilingual-cased",
    "distil-mbert": "distilbert-base-multilingual-cased",
    "xlm-roberta-large": "xlm-roberta-large",
}

DEFAULT_TASK_MODELS = ["xlm-roberta-base", "muril-base", "mbert-base"]

PATHS = PathConfig()
BASE_TRAIN_CONFIG = TrainConfig()

print("Path configuration:")
for f in ["task1_train", "task1_val", "task1_test", "task1_test_labeled",
          "task2_train", "task2_val", "task2_test", "task2_test_labeled",
          "task3_train", "task3_val", "task3_test", "task3_test_labeled"]:
    p = getattr(PATHS, f)
    print(f"  {f}: {p} | exists={p.exists()}")

print(f"\nOutput root: {PATHS.output_root}")
print(f"\nDefault models to train per task: {DEFAULT_TASK_MODELS}")

Path configuration:
  task1_train: C:\Users\VGST\Desktop\FIRE-2026\DravidianCodeMix 2026\SUBTASK 1\train.csv | exists=True
  task1_val: C:\Users\VGST\Desktop\FIRE-2026\DravidianCodeMix 2026\SUBTASK 1\validation.csv | exists=True
  task1_test: C:\Users\VGST\Desktop\FIRE-2026\DravidianCodeMix 2026\SUBTASK 1\test.csv | exists=True
  task1_test_labeled: C:\Users\VGST\Desktop\FIRE-2026\DravidianCodeMix 2026\SUBTASK 1\test_labeled.csv | exists=True
  task2_train: C:\Users\VGST\Desktop\FIRE-2026\DravidianCodeMix 2026\SUBTASK 2\train_multitask.csv | exists=True
  task2_val: C:\Users\VGST\Desktop\FIRE-2026\DravidianCodeMix 2026\SUBTASK 2\dev_multitask.csv | exists=True
  task2_test: C:\Users\VGST\Desktop\FIRE-2026\DravidianCodeMix 2026\SUBTASK 2\multitask_test_withoutlabel.csv | exists=True
  task2_test_labeled: C:\Users\VGST\Desktop\FIRE-2026\DravidianCodeMix 2026\SUBTASK 2\test with label.csv | exists=True
  task3_train: C:\Users\VGST\Desktop\FIRE-2026\DravidianCodeMix 2026\SUBTASK 3\Kannada_

In [ ]:
URL_PATTERN = re.compile(r"https?://\S+|www\.\S+")
HTML_TAG_PATTERN = re.compile(r"<.*?>")
MENTION_PATTERN = re.compile(r"@\w+")
HASHTAG_PATTERN = re.compile(r"#(\w+)")
EMOJI_PATTERN = re.compile(
    "["
    "\U0001F600-\U0001F64F"
    "\U0001F300-\U0001F5FF"
    "\U0001F680-\U0001F6FF"
    "\U0001F1E0-\U0001F1FF"
    "\U00002700-\U000027BF"
    "\U0001F900-\U0001F9FF"
    "\U00002600-\U000026FF"
    "\U0001FA70-\U0001FAFF"
    "]+",
    flags=re.UNICODE,
)
REPEATED_CHAR_PATTERN = re.compile(r"(.)\1{2,}")
REPEATED_PUNCT_PATTERN = re.compile(r"([!?.,])\1+")
MULTI_SPACE_PATTERN = re.compile(r"\s+")


def clean_text(
    text: str,
    remove_url: bool = True,
    remove_html: bool = True,
    remove_mentions: bool = True,
    handle_hashtags: bool = True,
    remove_emoji: bool = False,
    normalize_repeated_chars: bool = True,
    normalize_repeated_punct: bool = True,
    lowercase_latin_only: bool = True,
    unicode_normalize: bool = True,
) -> str:
    if not isinstance(text, str):
        return ""

    if unicode_normalize:
        text = unicodedata.normalize("NFKC", text)

    text = html_lib.unescape(text)

    if remove_html:
        text = HTML_TAG_PATTERN.sub(" ", text)

    if remove_url:
        text = URL_PATTERN.sub(" ", text)

    if remove_mentions:
        text = MENTION_PATTERN.sub(" ", text)

    if handle_hashtags:
        text = HASHTAG_PATTERN.sub(r"\1", text)

    if remove_emoji:
        text = EMOJI_PATTERN.sub(" ", text)

    if normalize_repeated_chars:
        text = REPEATED_CHAR_PATTERN.sub(r"\1\1", text)

    if normalize_repeated_punct:
        text = REPEATED_PUNCT_PATTERN.sub(r"\1", text)

    if lowercase_latin_only:
        text = "".join(
            c.lower() if c.isascii() and c.isalpha() else c for c in text
        )

    text = MULTI_SPACE_PATTERN.sub(" ", text).strip()

    return text


def preprocess_dataframe(df: pd.DataFrame, text_col: str, **clean_kwargs) -> pd.DataFrame:
    df = df.copy()
    df[text_col] = df[text_col].astype(str).apply(lambda x: clean_text(x, **clean_kwargs))
    df = df[df[text_col].str.strip().str.len() > 0].reset_index(drop=True)
    return df


def preprocess_dataframe_keep_all(df: pd.DataFrame, text_col: str, **clean_kwargs) -> pd.DataFrame:
    df = df.copy()
    df[text_col] = df[text_col].astype(str).apply(lambda x: clean_text(x, **clean_kwargs))
    empty_mask = df[text_col].str.strip().str.len() == 0
    if empty_mask.sum() > 0:
        print(f"WARNING: {empty_mask.sum()} test rows became empty after cleaning. "
              f"Keeping them with placeholder text so row count/order is preserved.")
        df.loc[empty_mask, text_col] = "[EMPTY]"
    return df


def build_label_encoders(label_space: Dict[str, List[str]]) -> Dict[str, LabelEncoder]:
    encoders = {}
    for level, classes in label_space.items():
        le = LabelEncoder()
        le.fit(classes)
        encoders[level] = le
    return encoders


sample_texts = [
    "Idhu semaaaa nalla iருக்குுு!!!! 😂😂😂 check http://example.com @user #trending",
    "  Multiple   spaces   and <b>html</b> tags here  ",
]
for t in sample_texts:
    print(f"BEFORE: {t}")
    print(f"AFTER:  {clean_text(t)}\n")

BEFORE: Idhu semaaaa nalla iருக்குுு!!!! 😂😂😂 check http://example.com @user #trending
AFTER:  idhu semaa nalla iருக்குு! 😂😂 check trending

BEFORE:   Multiple   spaces   and <b>html</b> tags here  
AFTER:  multiple spaces and html tags here



In [ ]:
def get_scheduler(scheduler_type, optimizer, num_warmup_steps, num_training_steps):
    if scheduler_type == "cosine":
        return get_cosine_schedule_with_warmup(
            optimizer, num_warmup_steps=num_warmup_steps, num_training_steps=num_training_steps
        )
    return get_linear_schedule_with_warmup(
        optimizer, num_warmup_steps=num_warmup_steps, num_training_steps=num_training_steps
    )


def save_checkpoint(path, model, optimizer, scheduler, scaler, epoch, best_metric, config):
    torch.save({
        "epoch": epoch,
        "model_state_dict": model.state_dict(),
        "optimizer_state_dict": optimizer.state_dict(),
        "scheduler_state_dict": scheduler.state_dict(),
        "scaler_state_dict": scaler.state_dict() if scaler is not None else None,
        "best_metric": best_metric,
        "config": config,
    }, path)


def load_checkpoint(path, model, optimizer=None, scheduler=None, scaler=None):
    checkpoint = torch.load(path, map_location=DEVICE)
    model.load_state_dict(checkpoint["model_state_dict"])
    if optimizer is not None and checkpoint.get("optimizer_state_dict") is not None:
        optimizer.load_state_dict(checkpoint["optimizer_state_dict"])
    if scheduler is not None and checkpoint.get("scheduler_state_dict") is not None:
        scheduler.load_state_dict(checkpoint["scheduler_state_dict"])
    if scaler is not None and checkpoint.get("scaler_state_dict") is not None:
        scaler.load_state_dict(checkpoint["scaler_state_dict"])
    return checkpoint.get("epoch", 0), checkpoint.get("best_metric", -1.0)


class FocalLoss(nn.Module):
    def __init__(self, gamma: float = 2.0, weight: Optional[torch.Tensor] = None,
                 label_smoothing: float = 0.0):
        super().__init__()
        self.gamma = gamma
        self.weight = weight
        self.label_smoothing = label_smoothing

    def forward(self, logits, targets):
        ce_loss = F.cross_entropy(
            logits, targets, weight=self.weight,
            reduction="none", label_smoothing=self.label_smoothing,
        )
        pt = torch.exp(-ce_loss)
        focal_loss = ((1 - pt) ** self.gamma) * ce_loss
        return focal_loss.mean()


def build_loss_fn(loss_type: str, class_weights: Optional[torch.Tensor] = None,
                   focal_gamma: float = 2.0, label_smoothing: float = 0.0):
    if loss_type == "focal":
        return FocalLoss(gamma=focal_gamma, weight=class_weights, label_smoothing=label_smoothing)
    elif loss_type == "weighted_ce":
        return nn.CrossEntropyLoss(weight=class_weights, label_smoothing=label_smoothing)
    else:
        return nn.CrossEntropyLoss(label_smoothing=label_smoothing)


def compute_class_weights_tensor(labels: List[int], num_classes: int, device: torch.device) -> torch.Tensor:
    present_classes = np.unique(labels)
    weights = compute_class_weight(class_weight="balanced", classes=present_classes, y=labels)
    full_weights = np.ones(num_classes, dtype=np.float32)
    for cls, w in zip(present_classes, weights):
        full_weights[cls] = w
    return torch.tensor(full_weights, dtype=torch.float32, device=device)


def plot_confusion_matrix(y_true, y_pred, class_names, title, save_path):
    cm = confusion_matrix(y_true, y_pred, labels=list(range(len(class_names))))
    plt.figure(figsize=(max(6, len(class_names)), max(5, len(class_names) * 0.8)))
    sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", xticklabels=class_names, yticklabels=class_names)
    plt.title(title)
    plt.xlabel("Predicted")
    plt.ylabel("True")
    plt.xticks(rotation=45, ha="right")
    plt.tight_layout()
    plt.savefig(save_path, dpi=100)
    plt.close()


def plot_learning_curves(train_losses, val_losses, title, save_path):
    plt.figure(figsize=(8, 5))
    plt.plot(train_losses, label="Train Loss", marker="o")
    plt.plot(val_losses, label="Val Loss", marker="o")
    plt.title(title)
    plt.xlabel("Epoch")
    plt.ylabel("Loss")
    plt.legend()
    plt.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.savefig(save_path, dpi=100)
    plt.close()


def hard_vote_ensemble(prob_dict_list: List[np.ndarray]) -> np.ndarray:
    preds = np.stack([np.argmax(p, axis=-1) for p in prob_dict_list], axis=0)
    num_classes = prob_dict_list[0].shape[-1]
    n_samples = preds.shape[1]
    final_preds = np.zeros(n_samples, dtype=int)
    for i in range(n_samples):
        counts = np.bincount(preds[:, i], minlength=num_classes)
        final_preds[i] = np.argmax(counts)
    return final_preds


def soft_vote_ensemble(prob_dict_list: List[np.ndarray], weights: Optional[List[float]] = None) -> np.ndarray:
    if weights is None:
        weights = [1.0 / len(prob_dict_list)] * len(prob_dict_list)
    weights = np.array(weights) / np.sum(weights)
    avg_probs = np.zeros_like(prob_dict_list[0])
    for w, p in zip(weights, prob_dict_list):
        avg_probs += w * p
    return avg_probs


def rank_average_ensemble(prob_dict_list: List[np.ndarray]) -> np.ndarray:
    n_models = len(prob_dict_list)
    n_samples, n_classes = prob_dict_list[0].shape
    rank_sum = np.zeros((n_samples, n_classes))
    for p in prob_dict_list:
        ranks = np.argsort(np.argsort(p, axis=-1), axis=-1)
        rank_sum += ranks
    rank_sum /= n_models
    return rank_sum


def logit_average_ensemble(prob_dict_list: List[np.ndarray], eps: float = 1e-9) -> np.ndarray:
    logit_sum = np.zeros_like(prob_dict_list[0])
    for p in prob_dict_list:
        p_clipped = np.clip(p, eps, 1 - eps)
        logit_sum += np.log(p_clipped / (1 - p_clipped))
    avg_logit = logit_sum / len(prob_dict_list)
    exp_logit = np.exp(avg_logit)
    return exp_logit / exp_logit.sum(axis=-1, keepdims=True)


def optimize_ensemble_weights(prob_dict_list: List[np.ndarray], y_true: List[int],
                               n_trials: int = 30) -> Tuple[List[float], float]:
    n_models = len(prob_dict_list)
    best_weights = [1.0 / n_models] * n_models
    best_f1 = f1_score(y_true, np.argmax(soft_vote_ensemble(prob_dict_list, best_weights), axis=-1),
                        average="macro", zero_division=0)

    rng = np.random.RandomState(GLOBAL_SEED)
    for _ in range(n_trials):
        raw = rng.dirichlet(np.ones(n_models))
        preds = np.argmax(soft_vote_ensemble(prob_dict_list, raw.tolist()), axis=-1)
        f1 = f1_score(y_true, preds, average="macro", zero_division=0)
        if f1 > best_f1:
            best_f1 = f1
            best_weights = raw.tolist()

    return best_weights, best_f1


def optimize_threshold_for_macro_f1(y_true: List[int], hate_probs: List[float],
                                     hate_class_id: int) -> Tuple[float, float]:
    thresholds = np.arange(0.05, 0.96, 0.01)
    best_threshold = 0.5
    best_f1 = -1.0

    for t in thresholds:
        preds = [hate_class_id if p >= t else (1 - hate_class_id) for p in hate_probs]
        f1 = f1_score(y_true, preds, average="macro", zero_division=0)
        if f1 > best_f1:
            best_f1 = f1
            best_threshold = t

    return best_threshold, best_f1


print("Shared training, evaluation, and ensemble utility functions defined")

Shared training, evaluation, and ensemble utility functions defined


Task 1 — Hierarchical Gender-aware Tamil Hate Speech Detection

In [ ]:
def load_task1_data():
    train_df = pd.read_csv(PATHS.task1_train)
    val_df = pd.read_csv(PATHS.task1_val)
    test_df = pd.read_csv(PATHS.task1_test)
    return train_df, val_df, test_df


task1_train_raw, task1_val_raw, task1_test_raw = load_task1_data()

print("TRAIN shape:", task1_train_raw.shape)
print("VAL shape:", task1_val_raw.shape)
print("TEST shape:", task1_test_raw.shape)
print("\nTRAIN columns:", list(task1_train_raw.columns))
print(task1_train_raw.head(3).to_string())

TRAIN shape: (4255, 7)
VAL shape: (532, 7)
TEST shape: (532, 2)

TRAIN columns: ['Input_ID', 'Input_Text', 'Level1_Class', 'Level2_Target', 'Level3_Category', 'Level4_Subcategory', 'Level5_Bias']
   Input_ID                                                                                                                                                                                                                                                                                               Input_Text Level1_Class       Level2_Target Level3_Category          Level4_Subcategory Level5_Bias
0      2815  சரி நீ தேவிடியா பையன் தான் ஆரியனாள் கற்பழிக்க பட்டதுதான் இந்து மதம்னு சொல்லுறேன் இல்லனு எப்புடி மறுப்ப சாதிகள் இல்லையடி பாப்பா என பாரதி பாடுறான் அவனும் ஒரு பாப்பான் தான் அதுக்கு என்ன சொல்றே போங்கடா போங்கடா போய் முதல்ல இந்து யாருன்னு முடிவு பண்ணிட்டு வா அப்புறமா  ஆர்எஸ்எஸ்  பீஜேபீ யை வளர்க்கலாம்         HATE  No Target- Profane        IMPLICIT                     Sarcasm         Nil
1      2

In [ ]:
TASK1_COLUMN_MAP = {
    "text": "Input_Text",
    "level1": "Level1_Class",
    "level2": "Level2_Target",
    "level3": "Level3_Category",
    "level4": "Level4_Subcategory",
    "level5": "Level5_Bias",
}

LEVEL1_NORMALIZE = {
    "HATE": "Hate",
    "NON-HATE": "Non-Hate",
}
LEVEL1_DROP_VALUES = {"Nil"}

LEVEL2_NORMALIZE = {
    "No Target- Profane": "Profane-Only",
    "Men- no profane": "Hate against Men",
    "Hate against Men": "Hate against Men",
    "Hate against Women": "Hate against Women",
    "Hate against LGBTQ+": "Hate against LGBTQ+",
    "Nil": "Nil",
}

LEVEL3_NORMALIZE = {
    "EXPLICIT": "Explicit",
    "IMPLICIT": "Implicit",
    "Nil": "Nil",
}

LEVEL4_SUBCATEGORY_NORMALIZE = {
    "Physical / Sexual Threats": "Physical/Sexual Threats",
    "Sexual Threat": "Physical/Sexual Threats",
    "Transphobic Attacks": "Homophobic/Transphobic Attacks",
    "Homophobic Slurs": "Homophobic/Transphobic Attacks",
    "Undermining / Mocking Masculinity": "Masculinity Attacks",
    "Steriotype": "Stereotype",
}

LEVEL5_NORMALIZE = {
    "Religional": "Religion",
}


def normalize_level1(value: str) -> Optional[str]:
    value = value.strip()
    if value in LEVEL1_DROP_VALUES:
        return None
    return LEVEL1_NORMALIZE.get(value, value)


def normalize_level2(value: str) -> str:
    value = value.strip()
    return LEVEL2_NORMALIZE.get(value, value)


def normalize_level3(value: str) -> str:
    value = value.strip()
    return LEVEL3_NORMALIZE.get(value, value)


def normalize_level4(value: str) -> str:
    value = value.strip()
    if value == "Nil":
        return "Nil"
    if "-" in value:
        subcategory = value.split("-", 1)[1].strip()
    else:
        subcategory = value
    return LEVEL4_SUBCATEGORY_NORMALIZE.get(subcategory, subcategory)


def normalize_level5(value: str) -> str:
    value = value.strip()
    return LEVEL5_NORMALIZE.get(value, value)


TASK1_LABEL_SPACE = {
    "level1": ["Non-Hate", "Hate"],
    "level2": [
        "Nil", "Hate against Women", "Hate against Men",
        "Hate against LGBTQ+", "Profane-Only",
    ],
    "level3": ["Nil", "Explicit", "Implicit"],
    "level4": [
        "Nil", "Character assassination", "Verbal Abusive", "Slut-shaming",
        "Body shaming", "Mocking Identity", "Moral policing", "Victim Blaming",
        "Sexual Objectification", "Physical/Sexual Threats",
        "Homophobic/Transphobic Attacks", "Masculinity Attacks", "Sarcasm",
        "Dismissal", "Pathologizing", "Misogyny", "Stereotype", "Exclusion",
    ],
    "level5": ["Nil", "Gender", "Religion", "Caste", "Geography", "Age", "Regional"],
}

TASK1_LEVEL_PARENT = {
    "level2": "level1", "level3": "level2", "level4": "level3", "level5": "level2",
}

TASK1_ENCODERS = build_label_encoders(TASK1_LABEL_SPACE)

for level, le in TASK1_ENCODERS.items():
    print(f"{level}: {list(le.classes_)}")

level1: ['Hate', 'Non-Hate']
level2: ['Hate against LGBTQ+', 'Hate against Men', 'Hate against Women', 'Nil', 'Profane-Only']
level3: ['Explicit', 'Implicit', 'Nil']
level4: ['Body shaming', 'Character assassination', 'Dismissal', 'Exclusion', 'Homophobic/Transphobic Attacks', 'Masculinity Attacks', 'Misogyny', 'Mocking Identity', 'Moral policing', 'Nil', 'Pathologizing', 'Physical/Sexual Threats', 'Sarcasm', 'Sexual Objectification', 'Slut-shaming', 'Stereotype', 'Verbal Abusive', 'Victim Blaming']
level5: ['Age', 'Caste', 'Gender', 'Geography', 'Nil', 'Regional', 'Religion']


In [ ]:
def validate_and_prepare_task1(df: pd.DataFrame, column_map: Dict[str, str],
                                encoders: Dict[str, LabelEncoder]) -> pd.DataFrame:
    df = df.copy()
    text_col = column_map["text"]

    if text_col not in df.columns:
        raise KeyError(f"Text column '{text_col}' not found. Available: {list(df.columns)}")

    df = preprocess_dataframe(df, text_col)
    df = df.rename(columns={text_col: "text"})

    normalize_fns = {
        "level1": normalize_level1, "level2": normalize_level2,
        "level3": normalize_level3, "level4": normalize_level4,
        "level5": normalize_level5,
    }

    for level in ["level1", "level2", "level3", "level4", "level5"]:
        col = column_map[level]
        if col not in df.columns:
            raise KeyError(f"Label column '{col}' for {level} not found. Available: {list(df.columns)}")
        df[col] = df[col].astype(str).apply(normalize_fns[level])

    before_drop = len(df)
    df = df[df[column_map["level1"]].notna()].reset_index(drop=True)
    dropped_l1 = before_drop - len(df)
    if dropped_l1 > 0:
        print(f"Dropped {dropped_l1} rows with unresolved Level1 label (raw value 'Nil')")

    keep_cols = ["text"]
    for level in ["level1", "level2", "level3", "level4", "level5"]:
        col = column_map[level]
        valid_classes = set(encoders[level].classes_)
        invalid_mask = ~df[col].isin(valid_classes)
        if invalid_mask.sum() > 0:
            print(f"WARNING: {invalid_mask.sum()} rows in {level} outside label space: "
                  f"{sorted(df.loc[invalid_mask, col].unique())[:10]}")
            df = df[~invalid_mask]
        df[f"{level}_id"] = encoders[level].transform(df[col])
        keep_cols.append(f"{level}_id")

    df = df.reset_index(drop=True)
    return df[keep_cols]


def validate_and_prepare_task1_test(df: pd.DataFrame, column_map: Dict[str, str],
                                     id_column_name: Optional[str] = None) -> pd.DataFrame:
    df = df.copy()
    text_col = column_map["text"]

    if text_col not in df.columns:
        raise KeyError(f"Text column '{text_col}' not found. Available: {list(df.columns)}")

    df = preprocess_dataframe_keep_all(df, text_col)
    df = df.rename(columns={text_col: "text"})

    out_cols = ["text"]
    if id_column_name and id_column_name in df.columns:
        out_cols = [id_column_name, "text"]
    return df[out_cols].reset_index(drop=True)


task1_train = validate_and_prepare_task1(task1_train_raw, TASK1_COLUMN_MAP, TASK1_ENCODERS)
task1_val = validate_and_prepare_task1(task1_val_raw, TASK1_COLUMN_MAP, TASK1_ENCODERS)
task1_test = validate_and_prepare_task1_test(task1_test_raw, TASK1_COLUMN_MAP, id_column_name="Input_ID")

print("Cleaned TRAIN shape:", task1_train.shape)
print("Cleaned VAL shape:", task1_val.shape)
print("TEST shape (must equal raw):", task1_test.shape, "vs raw:", task1_test_raw.shape)
assert len(task1_test) == len(task1_test_raw), "Test row count mismatch!"

for level in ["level1", "level2", "level3", "level4", "level5"]:
    print(f"\n{level} distribution (train):")
    print(task1_train[f"{level}_id"].value_counts().sort_index())

Dropped 9 rows with unresolved Level1 label (raw value 'Nil')
Dropped 1 rows with unresolved Level1 label (raw value 'Nil')
Cleaned TRAIN shape: (4246, 6)
Cleaned VAL shape: (531, 6)
TEST shape (must equal raw): (532, 2) vs raw: (532, 2)

level1 distribution (train):
level1_id
0    2231
1    2015
Name: count, dtype: int64

level2 distribution (train):
level2_id
0     144
1     617
2    1025
3    2043
4     417
Name: count, dtype: int64

level3 distribution (train):
level3_id
0     967
1    1258
2    2021
Name: count, dtype: int64

level4 distribution (train):
level4_id
0       77
1      250
2       82
3        2
4       25
5       55
6       15
7       32
8       63
9     2473
10       6
11      29
12     295
13      43
14     119
15      27
16     579
17      74
Name: count, dtype: int64

level5 distribution (train):
level5_id
0      13
1      87
2     215
3      40
4    3782
5       4
6     105
Name: count, dtype: int64


In [ ]:
class Task1Dataset(Dataset):
    def __init__(self, df: pd.DataFrame, tokenizer, max_len: int, is_test: bool = False):
        self.texts = df["text"].tolist()
        self.is_test = is_test
        self.tokenizer = tokenizer
        self.max_len = max_len

        if not is_test:
            self.level1 = df["level1_id"].tolist()
            self.level2 = df["level2_id"].tolist()
            self.level3 = df["level3_id"].tolist()
            self.level4 = df["level4_id"].tolist()
            self.level5 = df["level5_id"].tolist()

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        text = self.texts[idx]
        encoding = self.tokenizer(
            text, truncation=True, max_length=self.max_len, padding=False, return_attention_mask=True,
        )
        item = {"input_ids": encoding["input_ids"], "attention_mask": encoding["attention_mask"]}
        if not self.is_test:
            item["level1"] = self.level1[idx]
            item["level2"] = self.level2[idx]
            item["level3"] = self.level3[idx]
            item["level4"] = self.level4[idx]
            item["level5"] = self.level5[idx]
        return item


class DynamicPaddingCollator:
    def __init__(self, tokenizer, is_test: bool = False):
        self.tokenizer = tokenizer
        self.is_test = is_test

    def __call__(self, batch):
        input_ids = [torch.tensor(x["input_ids"], dtype=torch.long) for x in batch]
        attention_mask = [torch.tensor(x["attention_mask"], dtype=torch.long) for x in batch]

        max_len = max(len(x) for x in input_ids)
        pad_id = self.tokenizer.pad_token_id if self.tokenizer.pad_token_id is not None else 0

        padded_input_ids = torch.full((len(batch), max_len), pad_id, dtype=torch.long)
        padded_attention_mask = torch.zeros((len(batch), max_len), dtype=torch.long)

        for i, (ids, mask) in enumerate(zip(input_ids, attention_mask)):
            length = len(ids)
            padded_input_ids[i, :length] = ids
            padded_attention_mask[i, :length] = mask

        output = {"input_ids": padded_input_ids, "attention_mask": padded_attention_mask}

        if not self.is_test:
            for level in ["level1", "level2", "level3", "level4", "level5"]:
                output[level] = torch.tensor([x[level] for x in batch], dtype=torch.long)

        return output


print("Task1Dataset and DynamicPaddingCollator defined")

Task1Dataset and DynamicPaddingCollator defined


In [ ]:
class HierarchicalHateSpeechModel(nn.Module):
    def __init__(self, model_name: str, label_space: Dict[str, List[str]],
                 dropout: float = 0.1, use_gradient_checkpointing: bool = True):
        super().__init__()
        self.config = AutoConfig.from_pretrained(model_name)
        self.encoder = AutoModel.from_pretrained(model_name)

        if use_gradient_checkpointing and hasattr(self.encoder, "gradient_checkpointing_enable"):
            self.encoder.gradient_checkpointing_enable()

        hidden_size = self.config.hidden_size
        self.dropout = nn.Dropout(dropout)

        num_l1 = len(label_space["level1"])
        num_l2 = len(label_space["level2"])
        num_l3 = len(label_space["level3"])
        num_l4 = len(label_space["level4"])
        num_l5 = len(label_space["level5"])

        self.head_l1 = nn.Linear(hidden_size, num_l1)
        self.head_l2 = nn.Linear(hidden_size + num_l1, num_l2)
        self.head_l3 = nn.Linear(hidden_size + num_l2, num_l3)
        self.head_l4 = nn.Linear(hidden_size + num_l3, num_l4)
        self.head_l5 = nn.Linear(hidden_size + num_l2, num_l5)

    def mean_pooling(self, token_embeddings, attention_mask):
        mask_expanded = attention_mask.unsqueeze(-1).expand(token_embeddings.size()).float()
        sum_embeddings = torch.sum(token_embeddings * mask_expanded, dim=1)
        sum_mask = torch.clamp(mask_expanded.sum(dim=1), min=1e-9)
        return sum_embeddings / sum_mask

    def forward(self, input_ids, attention_mask):
        outputs = self.encoder(input_ids=input_ids, attention_mask=attention_mask)
        last_hidden = outputs.last_hidden_state
        pooled = self.mean_pooling(last_hidden, attention_mask)
        pooled = self.dropout(pooled)

        logits_l1 = self.head_l1(pooled)
        prob_l1 = F.softmax(logits_l1, dim=-1)

        l2_input = torch.cat([pooled, prob_l1], dim=-1)
        logits_l2 = self.head_l2(l2_input)
        prob_l2 = F.softmax(logits_l2, dim=-1)

        l3_input = torch.cat([pooled, prob_l2], dim=-1)
        logits_l3 = self.head_l3(l3_input)
        prob_l3 = F.softmax(logits_l3, dim=-1)

        l4_input = torch.cat([pooled, prob_l3], dim=-1)
        logits_l4 = self.head_l4(l4_input)

        l5_input = torch.cat([pooled, prob_l2], dim=-1)
        logits_l5 = self.head_l5(l5_input)

        return {
            "level1": logits_l1, "level2": logits_l2, "level3": logits_l3,
            "level4": logits_l4, "level5": logits_l5,
        }


print("HierarchicalHateSpeechModel defined")

HierarchicalHateSpeechModel defined


In [ ]:
class HierarchicalLoss(nn.Module):
    def __init__(self, loss_fns: Dict[str, nn.Module],
                 level_weights: Optional[Dict[str, float]] = None,
                 parent_map: Optional[Dict[str, str]] = None,
                 nil_ids: Optional[Dict[str, int]] = None):
        super().__init__()
        self.loss_fns = loss_fns
        self.level_weights = level_weights or {
            "level1": 1.0, "level2": 1.0, "level3": 0.8, "level4": 0.8, "level5": 0.6,
        }
        self.parent_map = parent_map or {}
        self.nil_ids = nil_ids or {}

    def forward(self, logits: Dict[str, torch.Tensor], targets: Dict[str, torch.Tensor]) -> Tuple[torch.Tensor, Dict[str, float]]:
        total_loss = 0.0
        loss_dict = {}

        for level in ["level1", "level2", "level3", "level4", "level5"]:
            level_logits = logits[level]
            level_targets = targets[level]

            if level in self.parent_map:
                parent_level = self.parent_map[level]
                parent_targets = targets[parent_level]
                nil_id = self.nil_ids.get(parent_level, 0)
                mask = (parent_targets != nil_id)
                if mask.sum() > 0:
                    loss = self.loss_fns[level](level_logits[mask], level_targets[mask])
                else:
                    loss = torch.tensor(0.0, device=level_logits.device)
            else:
                loss = self.loss_fns[level](level_logits, level_targets)

            weighted_loss = self.level_weights[level] * loss
            total_loss = total_loss + weighted_loss
            loss_dict[level] = loss.item() if torch.is_tensor(loss) else loss

        return total_loss, loss_dict


print("HierarchicalLoss defined")

HierarchicalLoss defined


In [ ]:
def train_one_epoch_task1(model, loader, optimizer, scheduler, scaler, loss_fn,
                           grad_accum_steps, max_grad_norm, device, use_amp):
    model.train()
    total_loss = 0.0
    num_batches = 0
    optimizer.zero_grad(set_to_none=True)

    for step, batch in enumerate(loader):
        try:
            input_ids = batch["input_ids"].to(device, non_blocking=True)
            attention_mask = batch["attention_mask"].to(device, non_blocking=True)
            targets = {
                level: batch[level].to(device, non_blocking=True)
                for level in ["level1", "level2", "level3", "level4", "level5"]
            }

            with autocast(device_type=device.type, enabled=use_amp):
                logits = model(input_ids, attention_mask)
                loss, loss_dict = loss_fn(logits, targets)
                loss = loss / grad_accum_steps

            if use_amp:
                scaler.scale(loss).backward()
            else:
                loss.backward()

            if (step + 1) % grad_accum_steps == 0:
                if use_amp:
                    scaler.unscale_(optimizer)
                    torch.nn.utils.clip_grad_norm_(model.parameters(), max_grad_norm)
                    scaler.step(optimizer)
                    scaler.update()
                else:
                    torch.nn.utils.clip_grad_norm_(model.parameters(), max_grad_norm)
                    optimizer.step()
                optimizer.zero_grad(set_to_none=True)
                if scheduler is not None:
                    scheduler.step()

            total_loss += loss.item() * grad_accum_steps
            num_batches += 1

            del input_ids, attention_mask, targets, logits, loss

        except torch.cuda.OutOfMemoryError:
            print(f"OOM at step {step}. Clearing cache and skipping batch.")
            optimizer.zero_grad(set_to_none=True)
            clear_memory()
            continue

    if (num_batches % grad_accum_steps) != 0:
        if use_amp:
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_grad_norm)
            scaler.step(optimizer)
            scaler.update()
        else:
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_grad_norm)
            optimizer.step()
        optimizer.zero_grad(set_to_none=True)

    clear_memory()
    return total_loss / max(num_batches, 1)


print("train_one_epoch_task1 defined")

train_one_epoch_task1 defined


In [ ]:
@torch.no_grad()
def evaluate_task1(model, loader, loss_fn, device, use_amp, encoders):
    model.eval()
    total_loss = 0.0
    num_batches = 0

    all_preds = {level: [] for level in ["level1", "level2", "level3", "level4", "level5"]}
    all_targets = {level: [] for level in ["level1", "level2", "level3", "level4", "level5"]}
    all_probs = {level: [] for level in ["level1", "level2", "level3", "level4", "level5"]}

    for batch in loader:
        input_ids = batch["input_ids"].to(device, non_blocking=True)
        attention_mask = batch["attention_mask"].to(device, non_blocking=True)
        targets = {
            level: batch[level].to(device, non_blocking=True)
            for level in ["level1", "level2", "level3", "level4", "level5"]
        }

        with autocast(device_type=device.type, enabled=use_amp):
            logits = model(input_ids, attention_mask)
            loss, _ = loss_fn(logits, targets)

        total_loss += loss.item()
        num_batches += 1

        for level in ["level1", "level2", "level3", "level4", "level5"]:
            probs = F.softmax(logits[level].float(), dim=-1)
            preds = torch.argmax(probs, dim=-1)
            all_preds[level].extend(preds.cpu().numpy().tolist())
            all_targets[level].extend(targets[level].cpu().numpy().tolist())
            all_probs[level].extend(probs.cpu().numpy().tolist())

        del input_ids, attention_mask, targets, logits

    clear_memory()

    metrics = {}
    for level in ["level1", "level2", "level3", "level4", "level5"]:
        y_true = all_targets[level]
        y_pred = all_preds[level]
        acc = accuracy_score(y_true, y_pred)
        precision, recall, f1_macro, _ = precision_recall_fscore_support(
            y_true, y_pred, average="macro", zero_division=0
        )
        _, _, f1_weighted, _ = precision_recall_fscore_support(
            y_true, y_pred, average="weighted", zero_division=0
        )
        per_class_f1 = f1_score(y_true, y_pred, average=None, zero_division=0)

        metrics[level] = {
            "accuracy": acc, "precision_macro": precision, "recall_macro": recall,
            "f1_macro": f1_macro, "f1_weighted": f1_weighted,
            "per_class_f1": per_class_f1.tolist(),
            "y_true": y_true, "y_pred": y_pred, "probs": all_probs[level],
        }

    avg_loss = total_loss / max(num_batches, 1)
    overall_f1_macro = np.mean([metrics[level]["f1_macro"] for level in metrics])

    return avg_loss, metrics, overall_f1_macro


print("evaluate_task1 defined")

evaluate_task1 defined


In [ ]:
def run_task1_training(model_key: str, model_name: str, train_df, val_df,
                        train_config: TrainConfig, encoders: Dict[str, LabelEncoder],
                        resume: bool = True) -> Dict[str, Any]:
    print(f"\n{'='*70}\nTraining Task 1 model: {model_key} ({model_name})\n{'='*70}")
    set_seed(train_config.seed)
    clear_memory()

    ckpt_dir = PATHS.output_root / "checkpoints" / "task1"
    ckpt_dir.mkdir(parents=True, exist_ok=True)
    ckpt_path = ckpt_dir / f"{model_key}_best.pt"
    last_ckpt_path = ckpt_dir / f"{model_key}_last.pt"

    try:
        tokenizer = AutoTokenizer.from_pretrained(model_name)
    except Exception as e:
        print(f"Failed to load tokenizer for {model_name}: {e}")
        return None

    try:
        model = HierarchicalHateSpeechModel(model_name, TASK1_LABEL_SPACE, dropout=train_config.dropout).to(DEVICE)
    except torch.cuda.OutOfMemoryError:
        print(f"OOM while loading {model_name}. Skipping.")
        clear_memory()
        return None
    except Exception as e:
        print(f"Failed to load model {model_name}: {e}")
        return None

    train_dataset = Task1Dataset(train_df, tokenizer, train_config.max_seq_len)
    val_dataset = Task1Dataset(val_df, tokenizer, train_config.max_seq_len)

    train_collator = DynamicPaddingCollator(tokenizer, is_test=False)
    val_collator = DynamicPaddingCollator(tokenizer, is_test=False)

    train_loader = DataLoader(
        train_dataset, batch_size=train_config.batch_size, shuffle=True,
        collate_fn=train_collator, num_workers=NUM_WORKERS, pin_memory=PIN_MEMORY,
    )
    val_loader = DataLoader(
        val_dataset, batch_size=train_config.eval_batch_size, shuffle=False,
        collate_fn=val_collator, num_workers=NUM_WORKERS, pin_memory=PIN_MEMORY,
    )

    loss_fns = {}
    nil_ids = {}
    for level in ["level1", "level2", "level3", "level4", "level5"]:
        labels = train_df[f"{level}_id"].tolist()
        num_classes = len(encoders[level].classes_)
        class_weights = None
        if train_config.loss_type == "weighted_ce":
            class_weights = compute_class_weights_tensor(labels, num_classes, DEVICE)
        loss_fns[level] = build_loss_fn(
            train_config.loss_type, class_weights, train_config.focal_gamma, train_config.label_smoothing,
        )
        if "Nil" in encoders[level].classes_:
            nil_ids[level] = int(encoders[level].transform(["Nil"])[0])
        elif "Non-Hate" in encoders[level].classes_:
            nil_ids[level] = int(encoders[level].transform(["Non-Hate"])[0])
        else:
            nil_ids[level] = 0

    hierarchical_loss = HierarchicalLoss(loss_fns=loss_fns, parent_map=TASK1_LEVEL_PARENT, nil_ids=nil_ids)

    optimizer = AdamW(model.parameters(), lr=train_config.lr, weight_decay=train_config.weight_decay)

    num_update_steps_per_epoch = math.ceil(len(train_loader) / train_config.grad_accum_steps)
    num_training_steps = num_update_steps_per_epoch * train_config.epochs
    num_warmup_steps = int(num_training_steps * train_config.warmup_ratio)
    scheduler = get_scheduler(train_config.scheduler_type, optimizer, num_warmup_steps, num_training_steps)

    scaler = GradScaler(enabled=USE_AMP)

    start_epoch = 0
    best_f1 = -1.0
    if resume and last_ckpt_path.exists():
        start_epoch, best_f1 = load_checkpoint(last_ckpt_path, model, optimizer, scheduler, scaler)
        print(f"Resumed from epoch {start_epoch} with best F1 {best_f1:.4f}")

    train_losses, val_losses = [], []
    patience_counter = 0
    history = []

    for epoch in range(start_epoch, train_config.epochs):
        epoch_start = time.time()
        train_loss = train_one_epoch_task1(
            model, train_loader, optimizer, scheduler, scaler, hierarchical_loss,
            train_config.grad_accum_steps, train_config.max_grad_norm, DEVICE, USE_AMP,
        )
        val_loss, val_metrics, overall_f1 = evaluate_task1(model, val_loader, hierarchical_loss, DEVICE, USE_AMP, encoders)

        train_losses.append(train_loss)
        val_losses.append(val_loss)
        epoch_time = time.time() - epoch_start

        print(f"Epoch {epoch+1}/{train_config.epochs} | train_loss={train_loss:.4f} | "
              f"val_loss={val_loss:.4f} | overall_f1_macro={overall_f1:.4f} | time={epoch_time:.1f}s")
        for level in ["level1", "level2", "level3", "level4", "level5"]:
            print(f"  {level}: f1_macro={val_metrics[level]['f1_macro']:.4f} acc={val_metrics[level]['accuracy']:.4f}")

        history.append({
            "epoch": epoch + 1, "train_loss": train_loss, "val_loss": val_loss,
            "overall_f1_macro": overall_f1,
            "level_f1": {level: val_metrics[level]["f1_macro"] for level in val_metrics},
        })

        save_checkpoint(last_ckpt_path, model, optimizer, scheduler, scaler, epoch + 1, best_f1, train_config)

        if overall_f1 > best_f1:
            best_f1 = overall_f1
            patience_counter = 0
            save_checkpoint(ckpt_path, model, optimizer, scheduler, scaler, epoch + 1, best_f1, train_config)
            print(f"  New best model saved. F1: {best_f1:.4f}")
        else:
            patience_counter += 1
            if patience_counter >= train_config.early_stopping_patience:
                print(f"  Early stopping at epoch {epoch+1}")
                break

    plot_learning_curves(
        train_losses, val_losses, f"Task1 {model_key} Learning Curve",
        PATHS.output_root / "plots" / f"task1_{model_key}_learning_curve.png",
    )

    log_path = PATHS.output_root / "logs" / f"task1_{model_key}_history.json"
    with open(log_path, "w") as f:
        json.dump(history, f, indent=2)

    load_checkpoint(ckpt_path, model, None, None, None)
    final_val_loss, final_val_metrics, final_f1 = evaluate_task1(model, val_loader, hierarchical_loss, DEVICE, USE_AMP, encoders)

    for level in ["level1", "level2", "level3", "level4", "level5"]:
        plot_confusion_matrix(
            final_val_metrics[level]["y_true"], final_val_metrics[level]["y_pred"],
            list(encoders[level].classes_), f"Task1 {model_key} {level} Confusion Matrix",
            PATHS.output_root / "plots" / f"task1_{model_key}_{level}_cm.png",
        )

    result = {
        "model_key": model_key, "model_name": model_name, "best_f1_macro": best_f1,
        "final_val_metrics": {
            level: {k: v for k, v in final_val_metrics[level].items() if k != "probs"}
            for level in final_val_metrics
        },
        "checkpoint_path": str(ckpt_path), "tokenizer_name": model_name, "history": history,
    }

    tokenizer_save_path = ckpt_dir / f"{model_key}_tokenizer"
    tokenizer.save_pretrained(tokenizer_save_path)

    del model, optimizer, scheduler, train_loader, val_loader
    clear_memory()
    print_gpu_memory(f"after {model_key} cleanup")

    return result


print("run_task1_training defined")

run_task1_training defined


In [ ]:
task1_results = {}
task1_config = TrainConfig(
    max_seq_len=192, batch_size=8, eval_batch_size=16, grad_accum_steps=4,
    epochs=5, lr=2e-5, loss_type="weighted_ce", scheduler_type="linear",
)

for model_key in DEFAULT_TASK_MODELS:
    model_name = MODEL_REGISTRY[model_key]
    result = run_task1_training(
        model_key=model_key, model_name=model_name, train_df=task1_train, val_df=task1_val,
        train_config=task1_config, encoders=TASK1_ENCODERS, resume=True,
    )
    if result is not None:
        task1_results[model_key] = result
    clear_memory()

print("\n\nTask 1 training summary:")
for key, res in task1_results.items():
    print(f"  {key}: best_f1_macro={res['best_f1_macro']:.4f}")


Training Task 1 model: xlm-roberta-base (xlm-roberta-base)
Resumed from epoch 5 with best F1 0.3524
[after xlm-roberta-base cleanup] GPU allocated: 0.01 GB | reserved: 0.02 GB

Training Task 1 model: muril-base (google/muril-base-cased)
Failed to load model google/muril-base-cased: Due to a serious vulnerability issue in `torch.load`, even with `weights_only=True`, we now require users to upgrade torch to at least v2.6 in order to use the function. This version restriction does not apply when loading files with safetensors.
See the vulnerability report here https://nvd.nist.gov/vuln/detail/CVE-2025-32434

Training Task 1 model: mbert-base (bert-base-multilingual-cased)
Resumed from epoch 5 with best F1 0.3479
[after mbert-base cleanup] GPU allocated: 0.01 GB | reserved: 0.02 GB


Task 1 training summary:
  xlm-roberta-base: best_f1_macro=0.3524
  mbert-base: best_f1_macro=0.3479


In [ ]:
@torch.no_grad()
def get_task1_predictions(model_key: str, model_name: str, checkpoint_path: Path,
                           tokenizer_path: Path, df: pd.DataFrame, max_seq_len: int,
                           eval_batch_size: int, has_labels: bool) -> Dict[str, np.ndarray]:
    clear_memory()
    tokenizer = AutoTokenizer.from_pretrained(tokenizer_path)

    model = HierarchicalHateSpeechModel(model_name, TASK1_LABEL_SPACE).to(DEVICE)
    load_checkpoint(checkpoint_path, model, None, None, None)
    model.eval()

    dataset = Task1Dataset(df, tokenizer, max_seq_len, is_test=not has_labels)
    collator = DynamicPaddingCollator(tokenizer, is_test=not has_labels)
    loader = DataLoader(
        dataset, batch_size=eval_batch_size, shuffle=False,
        collate_fn=collator, num_workers=NUM_WORKERS, pin_memory=PIN_MEMORY,
    )

    all_probs = {level: [] for level in ["level1", "level2", "level3", "level4", "level5"]}

    for batch in loader:
        input_ids = batch["input_ids"].to(DEVICE, non_blocking=True)
        attention_mask = batch["attention_mask"].to(DEVICE, non_blocking=True)

        with autocast(device_type=DEVICE.type, enabled=USE_AMP):
            logits = model(input_ids, attention_mask)

        for level in ["level1", "level2", "level3", "level4", "level5"]:
            probs = F.softmax(logits[level].float(), dim=-1)
            all_probs[level].append(probs.cpu().numpy())

        del input_ids, attention_mask, logits

    for level in all_probs:
        all_probs[level] = np.concatenate(all_probs[level], axis=0)

    del model
    clear_memory()

    return all_probs


def apply_hierarchical_override(probs_by_level: Dict[str, np.ndarray],
                                 encoders: Dict[str, LabelEncoder]) -> Dict[str, np.ndarray]:
    final_preds = {}

    final_preds["level1"] = np.argmax(probs_by_level["level1"], axis=-1)
    level1_nil_id = int(encoders["level1"].transform(["Non-Hate"])[0])
    level1_is_nil = final_preds["level1"] == level1_nil_id

    level2_nil_id = int(encoders["level2"].transform(["Nil"])[0])
    level2_preds = np.argmax(probs_by_level["level2"], axis=-1)
    level2_preds = np.where(level1_is_nil, level2_nil_id, level2_preds)
    final_preds["level2"] = level2_preds
    level2_is_nil = level2_preds == level2_nil_id

    level3_nil_id = int(encoders["level3"].transform(["Nil"])[0])
    level3_preds = np.argmax(probs_by_level["level3"], axis=-1)
    level3_preds = np.where(level2_is_nil, level3_nil_id, level3_preds)
    final_preds["level3"] = level3_preds
    level3_is_nil = level3_preds == level3_nil_id

    level4_nil_id = int(encoders["level4"].transform(["Nil"])[0])
    level4_preds = np.argmax(probs_by_level["level4"], axis=-1)
    level4_preds = np.where(level3_is_nil, level4_nil_id, level4_preds)
    final_preds["level4"] = level4_preds

    level5_nil_id = int(encoders["level5"].transform(["Nil"])[0])
    level5_preds = np.argmax(probs_by_level["level5"], axis=-1)
    level5_preds = np.where(level2_is_nil, level5_nil_id, level5_preds)
    final_preds["level5"] = level5_preds

    return final_preds


print("get_task1_predictions and apply_hierarchical_override defined")

get_task1_predictions and apply_hierarchical_override defined


In [ ]:
task1_val_probs = {}
task1_test_probs = {}

for model_key, result in task1_results.items():
    print(f"Collecting predictions for {model_key}")
    ckpt_dir = PATHS.output_root / "checkpoints" / "task1"
    tokenizer_path = ckpt_dir / f"{model_key}_tokenizer"
    checkpoint_path = Path(result["checkpoint_path"])

    val_probs = get_task1_predictions(
        model_key, result["model_name"], checkpoint_path, tokenizer_path,
        task1_val, task1_config.max_seq_len, task1_config.eval_batch_size, has_labels=True,
    )
    test_probs = get_task1_predictions(
        model_key, result["model_name"], checkpoint_path, tokenizer_path,
        task1_test, task1_config.max_seq_len, task1_config.eval_batch_size, has_labels=False,
    )

    task1_val_probs[model_key] = val_probs
    task1_test_probs[model_key] = test_probs
    clear_memory()

print("Collected val/test probabilities for:", list(task1_val_probs.keys()))

Collected val/test probabilities for: ['xlm-roberta-base', 'mbert-base']


In [ ]:
def evaluate_ensembles_task1(val_probs_by_model: Dict[str, Dict[str, np.ndarray]],
                              val_df: pd.DataFrame, encoders: Dict[str, LabelEncoder]) -> Dict[str, Any]:
    model_keys = list(val_probs_by_model.keys())
    ensemble_results = {}

    for level in ["level1", "level2", "level3", "level4", "level5"]:
        y_true = val_df[f"{level}_id"].tolist()
        prob_list = [val_probs_by_model[mk][level] for mk in model_keys]

        strategies = {}
        hard_preds = hard_vote_ensemble(prob_list)
        strategies["hard_vote"] = {"preds": hard_preds, "f1_macro": f1_score(y_true, hard_preds, average="macro", zero_division=0)}

        soft_preds = np.argmax(soft_vote_ensemble(prob_list), axis=-1)
        strategies["soft_vote_uniform"] = {"preds": soft_preds, "f1_macro": f1_score(y_true, soft_preds, average="macro", zero_division=0)}

        rank_preds = np.argmax(rank_average_ensemble(prob_list), axis=-1)
        strategies["rank_average"] = {"preds": rank_preds, "f1_macro": f1_score(y_true, rank_preds, average="macro", zero_division=0)}

        logit_preds = np.argmax(logit_average_ensemble(prob_list), axis=-1)
        strategies["logit_average"] = {"preds": logit_preds, "f1_macro": f1_score(y_true, logit_preds, average="macro", zero_division=0)}

        opt_weights, opt_f1 = optimize_ensemble_weights(prob_list, y_true, n_trials=30)
        weighted_preds = np.argmax(soft_vote_ensemble(prob_list, opt_weights), axis=-1)
        strategies["weighted_optimized"] = {
            "preds": weighted_preds, "f1_macro": f1_score(y_true, weighted_preds, average="macro", zero_division=0),
            "weights": opt_weights,
        }

        for mk in model_keys:
            single_preds = np.argmax(val_probs_by_model[mk][level], axis=-1)
            strategies[f"single_{mk}"] = {"preds": single_preds, "f1_macro": f1_score(y_true, single_preds, average="macro", zero_division=0)}

        best_strategy = max(strategies.items(), key=lambda x: x[1]["f1_macro"])
        ensemble_results[level] = {
            "strategies": strategies, "best_strategy_name": best_strategy[0],
            "best_f1_macro": best_strategy[1]["f1_macro"],
        }

        print(f"\n{level} ensemble comparison:")
        for name, res in sorted(strategies.items(), key=lambda x: -x[1]["f1_macro"]):
            print(f"  {name}: f1_macro={res['f1_macro']:.4f}")
        print(f"  BEST: {best_strategy[0]} ({best_strategy[1]['f1_macro']:.4f})")

    return ensemble_results


task1_ensemble_results = evaluate_ensembles_task1(task1_val_probs, task1_val, TASK1_ENCODERS)


level1 ensemble comparison:
  soft_vote_uniform: f1_macro=0.7944
  logit_average: f1_macro=0.7944
  weighted_optimized: f1_macro=0.7944
  single_mbert-base: f1_macro=0.7774
  hard_vote: f1_macro=0.7676
  rank_average: f1_macro=0.7676
  single_xlm-roberta-base: f1_macro=0.7656
  BEST: soft_vote_uniform (0.7944)

level2 ensemble comparison:
  weighted_optimized: f1_macro=0.3611
  single_xlm-roberta-base: f1_macro=0.3588
  rank_average: f1_macro=0.3505
  logit_average: f1_macro=0.3499
  soft_vote_uniform: f1_macro=0.3497
  single_mbert-base: f1_macro=0.3456
  hard_vote: f1_macro=0.3422
  BEST: weighted_optimized (0.3611)

level3 ensemble comparison:
  weighted_optimized: f1_macro=0.3656
  soft_vote_uniform: f1_macro=0.3590
  logit_average: f1_macro=0.3590
  single_xlm-roberta-base: f1_macro=0.3567
  single_mbert-base: f1_macro=0.3532
  hard_vote: f1_macro=0.3376
  rank_average: f1_macro=0.3376
  BEST: weighted_optimized (0.3656)

level4 ensemble comparison:
  hard_vote: f1_macro=0.1145
 

In [ ]:
best_strategy_per_level = {
    level: task1_ensemble_results[level]["best_strategy_name"]
    for level in ["level1", "level2", "level3", "level4", "level5"]
}

submission_2_strategy = {level: "soft_vote_uniform" for level in ["level1", "level2", "level3", "level4", "level5"]}

best_single_model_t1 = max(task1_results.items(), key=lambda x: x[1]["best_f1_macro"])[0]
submission_3_strategy = {level: f"single_{best_single_model_t1}" for level in ["level1", "level2", "level3", "level4", "level5"]}

print("submission_1 strategy per level:", best_strategy_per_level)
print("submission_2 strategy: uniform soft-vote")
print("submission_3 strategy: best single model =", best_single_model_t1)

submission_1 strategy per level: {'level1': 'soft_vote_uniform', 'level2': 'weighted_optimized', 'level3': 'weighted_optimized', 'level4': 'hard_vote', 'level5': 'weighted_optimized'}
submission_2 strategy: uniform soft-vote
submission_3 strategy: best single model = xlm-roberta-base


In [ ]:
def build_task1_submission(test_probs_by_model: Dict[str, Dict[str, np.ndarray]],
                            strategy_per_level: Dict[str, str],
                            ensemble_results: Dict[str, Any],
                            encoders: Dict[str, LabelEncoder],
                            processed_test_df: pd.DataFrame,
                            id_column_name: str = "Input_ID") -> pd.DataFrame:
    model_keys = list(test_probs_by_model.keys())
    submission = pd.DataFrame()

    if id_column_name in processed_test_df.columns:
        submission["id"] = processed_test_df[id_column_name].values
    else:
        submission["id"] = np.arange(len(processed_test_df))

    for level in ["level1", "level2", "level3", "level4", "level5"]:
        strategy_name = strategy_per_level[level]
        prob_list = [test_probs_by_model[mk][level] for mk in model_keys]

        assert len(prob_list[0]) == len(submission), f"Prediction count mismatch for {level}"

        if strategy_name == "hard_vote":
            preds = hard_vote_ensemble(prob_list)
        elif strategy_name == "soft_vote_uniform":
            preds = np.argmax(soft_vote_ensemble(prob_list), axis=-1)
        elif strategy_name == "rank_average":
            preds = np.argmax(rank_average_ensemble(prob_list), axis=-1)
        elif strategy_name == "logit_average":
            preds = np.argmax(logit_average_ensemble(prob_list), axis=-1)
        elif strategy_name == "weighted_optimized":
            weights = ensemble_results[level]["strategies"]["weighted_optimized"]["weights"]
            preds = np.argmax(soft_vote_ensemble(prob_list, weights), axis=-1)
        elif strategy_name.startswith("single_"):
            mk = strategy_name.replace("single_", "")
            preds = np.argmax(test_probs_by_model[mk][level], axis=-1)
        else:
            preds = np.argmax(soft_vote_ensemble(prob_list), axis=-1)

        submission[level] = encoders[level].inverse_transform(preds)

    return submission


task1_submission_1 = build_task1_submission(
    task1_test_probs, best_strategy_per_level, task1_ensemble_results, TASK1_ENCODERS,
    task1_test, id_column_name="Input_ID",
)
task1_submission_2 = build_task1_submission(
    task1_test_probs, submission_2_strategy, task1_ensemble_results, TASK1_ENCODERS,
    task1_test, id_column_name="Input_ID",
)
task1_submission_3 = build_task1_submission(
    task1_test_probs, submission_3_strategy, task1_ensemble_results, TASK1_ENCODERS,
    task1_test, id_column_name="Input_ID",
)

sub_dir = PATHS.output_root / "submissions"
task1_submission_1.to_csv(sub_dir / "task1_submission_1.csv", index=False)
task1_submission_2.to_csv(sub_dir / "task1_submission_2.csv", index=False)
task1_submission_3.to_csv(sub_dir / "task1_submission_3.csv", index=False)

print("Task 1 submissions saved. Verifying ID alignment:")
for f in ["task1_submission_1.csv", "task1_submission_2.csv", "task1_submission_3.csv"]:
    sub = pd.read_csv(sub_dir / f)
    ids_match = sub["id"].tolist() == task1_test_raw["Input_ID"].tolist()
    print(f"  {f}: rows={len(sub)}, IDs_match_raw_order={ids_match}")
print("\nsubmission_1 preview:")
print(task1_submission_1.head())

Task 1 submissions saved. Verifying ID alignment:
  task1_submission_1.csv: rows=532, IDs_match_raw_order=True
  task1_submission_2.csv: rows=532, IDs_match_raw_order=True
  task1_submission_3.csv: rows=532, IDs_match_raw_order=True

submission_1 preview:
     id    level1              level2    level3                   level4  \
0  2604      Hate    Hate against Men  Explicit           Verbal Abusive   
1  4803  Non-Hate  Hate against Women  Implicit  Character assassination   
2  3512  Non-Hate        Profane-Only  Implicit                      Nil   
3  4083  Non-Hate        Profane-Only  Implicit                      Nil   
4  3426  Non-Hate    Hate against Men  Implicit                  Sarcasm   

  level5  
0    Nil  
1    Nil  
2    Nil  
3    Nil  
4    Nil  


Task 2 — Multi-task Hope Speech Detection

In [ ]:
def load_task2_data():
    train_df = pd.read_csv(PATHS.task2_train)
    val_df = pd.read_csv(PATHS.task2_val)
    test_df = pd.read_csv(PATHS.task2_test)
    return train_df, val_df, test_df


task2_train_raw, task2_val_raw, task2_test_raw = load_task2_data()

print("TRAIN shape:", task2_train_raw.shape)
print("VAL shape:", task2_val_raw.shape)
print("TEST shape:", task2_test_raw.shape)
print("\nTRAIN columns:", list(task2_train_raw.columns))
print(task2_train_raw.head(3).to_string())

TRAIN shape: (3513, 4)
VAL shape: (753, 4)
TEST shape: (744, 2)

TRAIN columns: ['Id', 'text', 'Coarse-Grained', 'Fine-Grained Hope']
   Id                                    text    Coarse-Grained Fine-Grained Hope
0   1  Fast time kannir batina video mare😭😭😭😭      Blended Hope      Hopelessness
1   2          Av daada Anna background sound      Blended Hope      Hopelessness
2   3           wow nice comedy super athindh  Encouraging Hope    Inspiring Hope


In [ ]:
TASK2_COLUMN_MAP = {
    "text": "text",
    "tone": "Coarse-Grained",
    "hope_type": "Fine-Grained Hope",
}

TONE_NORMALIZE = {
    "Blended Hope": "Blended",
    "Encouraging Hope": "Encouraging",
    "Discouraging Hope": "Discouraging",
}


def normalize_tone(value: str) -> str:
    value = value.strip()
    return TONE_NORMALIZE.get(value, value)


def normalize_hope_type(value: str) -> str:
    return value.strip()


TASK2_LABEL_SPACE = {
    "tone": ["Encouraging", "Discouraging", "Blended"],
    "hope_type": ["Optimistic Hope", "Realistic Hope", "Inspiring Hope", "Fading Hope", "Hopelessness"],
}

TASK2_ENCODERS = build_label_encoders(TASK2_LABEL_SPACE)

for level, le in TASK2_ENCODERS.items():
    print(f"{level}: {list(le.classes_)}")

TASK2_LOSS_WEIGHT_CONFIGS = {
    "equal": {"tone": 1.0, "hope_type": 1.0},
    "tone_heavy": {"tone": 1.5, "hope_type": 0.5},
    "hope_type_heavy": {"tone": 0.5, "hope_type": 1.5},
    "balanced_070_030": {"tone": 0.7, "hope_type": 0.3},
}

tone: ['Blended', 'Discouraging', 'Encouraging']
hope_type: ['Fading Hope', 'Hopelessness', 'Inspiring Hope', 'Optimistic Hope', 'Realistic Hope']


In [ ]:
def validate_and_prepare_task2(df: pd.DataFrame, column_map: Dict[str, str],
                                encoders: Dict[str, LabelEncoder]) -> pd.DataFrame:
    df = df.copy()
    text_col = column_map["text"]

    if text_col not in df.columns:
        raise KeyError(f"Text column '{text_col}' not found. Available: {list(df.columns)}")

    df = preprocess_dataframe(df, text_col)
    df = df.rename(columns={text_col: "text"})

    normalize_fns = {"tone": normalize_tone, "hope_type": normalize_hope_type}

    keep_cols = ["text"]
    for task in ["tone", "hope_type"]:
        col = column_map[task]
        if col not in df.columns:
            raise KeyError(f"Label column '{col}' for {task} not found. Available: {list(df.columns)}")
        df[col] = df[col].astype(str).apply(normalize_fns[task])
        valid_classes = set(encoders[task].classes_)
        invalid_mask = ~df[col].isin(valid_classes)
        if invalid_mask.sum() > 0:
            print(f"WARNING: {invalid_mask.sum()} rows in {task} outside label space: "
                  f"{sorted(df.loc[invalid_mask, col].unique())[:10]}")
            df = df[~invalid_mask]
        df[f"{task}_id"] = encoders[task].transform(df[col])
        keep_cols.append(f"{task}_id")

    df = df.reset_index(drop=True)
    return df[keep_cols]


def validate_and_prepare_task2_test(df: pd.DataFrame, column_map: Dict[str, str],
                                     id_column_name: Optional[str] = None) -> pd.DataFrame:
    df = df.copy()
    text_col = column_map["text"]

    if text_col not in df.columns:
        raise KeyError(f"Text column '{text_col}' not found. Available: {list(df.columns)}")

    df = preprocess_dataframe_keep_all(df, text_col)
    df = df.rename(columns={text_col: "text"})

    out_cols = ["text"]
    if id_column_name and id_column_name in df.columns:
        out_cols = [id_column_name, "text"]
    return df[out_cols].reset_index(drop=True)


task2_train = validate_and_prepare_task2(task2_train_raw, TASK2_COLUMN_MAP, TASK2_ENCODERS)
task2_val = validate_and_prepare_task2(task2_val_raw, TASK2_COLUMN_MAP, TASK2_ENCODERS)
task2_test = validate_and_prepare_task2_test(task2_test_raw, TASK2_COLUMN_MAP, id_column_name="Id")

print("Cleaned TRAIN shape:", task2_train.shape)
print("Cleaned VAL shape:", task2_val.shape)
print("TEST shape (must equal raw):", task2_test.shape, "vs raw:", task2_test_raw.shape)
assert len(task2_test) == len(task2_test_raw), "Test row count mismatch!"

for task in ["tone", "hope_type"]:
    print(f"\n{task} distribution (train):")
    print(task2_train[f"{task}_id"].value_counts().sort_index())

Cleaned TRAIN shape: (3513, 3)
Cleaned VAL shape: (753, 3)
TEST shape (must equal raw): (744, 2) vs raw: (744, 2)

tone distribution (train):
tone_id
0     761
1     766
2    1986
Name: count, dtype: int64

hope_type distribution (train):
hope_type_id
0     283
1    1061
2    1222
3     408
4     539
Name: count, dtype: int64


In [ ]:
class Task2Dataset(Dataset):
    def __init__(self, df: pd.DataFrame, tokenizer, max_len: int, is_test: bool = False):
        self.texts = df["text"].tolist()
        self.is_test = is_test
        self.tokenizer = tokenizer
        self.max_len = max_len

        if not is_test:
            self.tone = df["tone_id"].tolist()
            self.hope_type = df["hope_type_id"].tolist()

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        text = self.texts[idx]
        encoding = self.tokenizer(
            text, truncation=True, max_length=self.max_len, padding=False, return_attention_mask=True,
        )
        item = {"input_ids": encoding["input_ids"], "attention_mask": encoding["attention_mask"]}
        if not self.is_test:
            item["tone"] = self.tone[idx]
            item["hope_type"] = self.hope_type[idx]
        return item


class Task2Collator:
    def __init__(self, tokenizer, is_test: bool = False):
        self.tokenizer = tokenizer
        self.is_test = is_test

    def __call__(self, batch):
        input_ids = [torch.tensor(x["input_ids"], dtype=torch.long) for x in batch]
        attention_mask = [torch.tensor(x["attention_mask"], dtype=torch.long) for x in batch]

        max_len = max(len(x) for x in input_ids)
        pad_id = self.tokenizer.pad_token_id if self.tokenizer.pad_token_id is not None else 0

        padded_input_ids = torch.full((len(batch), max_len), pad_id, dtype=torch.long)
        padded_attention_mask = torch.zeros((len(batch), max_len), dtype=torch.long)

        for i, (ids, mask) in enumerate(zip(input_ids, attention_mask)):
            length = len(ids)
            padded_input_ids[i, :length] = ids
            padded_attention_mask[i, :length] = mask

        output = {"input_ids": padded_input_ids, "attention_mask": padded_attention_mask}

        if not self.is_test:
            output["tone"] = torch.tensor([x["tone"] for x in batch], dtype=torch.long)
            output["hope_type"] = torch.tensor([x["hope_type"] for x in batch], dtype=torch.long)

        return output


print("Task2Dataset and Task2Collator defined")

Task2Dataset and Task2Collator defined


In [ ]:
class MultiTaskHopeSpeechModel(nn.Module):
    def __init__(self, model_name: str, label_space: Dict[str, List[str]],
                 dropout: float = 0.1, use_gradient_checkpointing: bool = True,
                 shared_projection_dim: int = 256):
        super().__init__()
        self.config = AutoConfig.from_pretrained(model_name)
        self.encoder = AutoModel.from_pretrained(model_name)

        if use_gradient_checkpointing and hasattr(self.encoder, "gradient_checkpointing_enable"):
            self.encoder.gradient_checkpointing_enable()

        hidden_size = self.config.hidden_size
        self.dropout = nn.Dropout(dropout)

        self.shared_projection = nn.Sequential(
            nn.Linear(hidden_size, shared_projection_dim), nn.GELU(), nn.Dropout(dropout),
        )

        num_tone = len(label_space["tone"])
        num_hope_type = len(label_space["hope_type"])

        self.tone_head = nn.Sequential(
            nn.Linear(shared_projection_dim, shared_projection_dim // 2), nn.GELU(), nn.Dropout(dropout),
            nn.Linear(shared_projection_dim // 2, num_tone),
        )
        self.hope_type_head = nn.Sequential(
            nn.Linear(shared_projection_dim, shared_projection_dim // 2), nn.GELU(), nn.Dropout(dropout),
            nn.Linear(shared_projection_dim // 2, num_hope_type),
        )

    def mean_pooling(self, token_embeddings, attention_mask):
        mask_expanded = attention_mask.unsqueeze(-1).expand(token_embeddings.size()).float()
        sum_embeddings = torch.sum(token_embeddings * mask_expanded, dim=1)
        sum_mask = torch.clamp(mask_expanded.sum(dim=1), min=1e-9)
        return sum_embeddings / sum_mask

    def forward(self, input_ids, attention_mask):
        outputs = self.encoder(input_ids=input_ids, attention_mask=attention_mask)
        last_hidden = outputs.last_hidden_state
        pooled = self.mean_pooling(last_hidden, attention_mask)
        pooled = self.dropout(pooled)
        shared_repr = self.shared_projection(pooled)
        return {"tone": self.tone_head(shared_repr), "hope_type": self.hope_type_head(shared_repr)}


class MultiTaskLoss(nn.Module):
    def __init__(self, loss_fns: Dict[str, nn.Module], task_weights: Dict[str, float]):
        super().__init__()
        self.loss_fns = loss_fns
        self.task_weights = task_weights

    def forward(self, logits, targets):
        total_loss = 0.0
        loss_dict = {}
        for task in ["tone", "hope_type"]:
            loss = self.loss_fns[task](logits[task], targets[task])
            total_loss = total_loss + self.task_weights[task] * loss
            loss_dict[task] = loss.item()
        return total_loss, loss_dict


print("MultiTaskHopeSpeechModel and MultiTaskLoss defined")

MultiTaskHopeSpeechModel and MultiTaskLoss defined


In [ ]:
def train_one_epoch_task2(model, loader, optimizer, scheduler, scaler, loss_fn,
                           grad_accum_steps, max_grad_norm, device, use_amp):
    model.train()
    total_loss = 0.0
    num_batches = 0
    optimizer.zero_grad(set_to_none=True)

    for step, batch in enumerate(loader):
        try:
            input_ids = batch["input_ids"].to(device, non_blocking=True)
            attention_mask = batch["attention_mask"].to(device, non_blocking=True)
            targets = {
                "tone": batch["tone"].to(device, non_blocking=True),
                "hope_type": batch["hope_type"].to(device, non_blocking=True),
            }

            with autocast(device_type=device.type, enabled=use_amp):
                logits = model(input_ids, attention_mask)
                loss, loss_dict = loss_fn(logits, targets)
                loss = loss / grad_accum_steps

            if use_amp:
                scaler.scale(loss).backward()
            else:
                loss.backward()

            if (step + 1) % grad_accum_steps == 0:
                if use_amp:
                    scaler.unscale_(optimizer)
                    torch.nn.utils.clip_grad_norm_(model.parameters(), max_grad_norm)
                    scaler.step(optimizer)
                    scaler.update()
                else:
                    torch.nn.utils.clip_grad_norm_(model.parameters(), max_grad_norm)
                    optimizer.step()
                optimizer.zero_grad(set_to_none=True)
                if scheduler is not None:
                    scheduler.step()

            total_loss += loss.item() * grad_accum_steps
            num_batches += 1

            del input_ids, attention_mask, targets, logits, loss

        except torch.cuda.OutOfMemoryError:
            print(f"OOM at step {step}. Clearing cache and skipping batch.")
            optimizer.zero_grad(set_to_none=True)
            clear_memory()
            continue

    if (num_batches % grad_accum_steps) != 0:
        if use_amp:
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_grad_norm)
            scaler.step(optimizer)
            scaler.update()
        else:
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_grad_norm)
            optimizer.step()
        optimizer.zero_grad(set_to_none=True)

    clear_memory()
    return total_loss / max(num_batches, 1)


@torch.no_grad()
def evaluate_task2(model, loader, loss_fn, device, use_amp):
    model.eval()
    total_loss = 0.0
    num_batches = 0

    all_preds = {"tone": [], "hope_type": []}
    all_targets = {"tone": [], "hope_type": []}
    all_probs = {"tone": [], "hope_type": []}

    for batch in loader:
        input_ids = batch["input_ids"].to(device, non_blocking=True)
        attention_mask = batch["attention_mask"].to(device, non_blocking=True)
        targets = {
            "tone": batch["tone"].to(device, non_blocking=True),
            "hope_type": batch["hope_type"].to(device, non_blocking=True),
        }

        with autocast(device_type=device.type, enabled=use_amp):
            logits = model(input_ids, attention_mask)
            loss, _ = loss_fn(logits, targets)

        total_loss += loss.item()
        num_batches += 1

        for task in ["tone", "hope_type"]:
            probs = F.softmax(logits[task].float(), dim=-1)
            preds = torch.argmax(probs, dim=-1)
            all_preds[task].extend(preds.cpu().numpy().tolist())
            all_targets[task].extend(targets[task].cpu().numpy().tolist())
            all_probs[task].extend(probs.cpu().numpy().tolist())

        del input_ids, attention_mask, targets, logits

    clear_memory()

    metrics = {}
    for task in ["tone", "hope_type"]:
        y_true = all_targets[task]
        y_pred = all_preds[task]
        acc = accuracy_score(y_true, y_pred)
        precision, recall, f1_macro, _ = precision_recall_fscore_support(y_true, y_pred, average="macro", zero_division=0)
        _, _, f1_weighted, _ = precision_recall_fscore_support(y_true, y_pred, average="weighted", zero_division=0)
        per_class_f1 = f1_score(y_true, y_pred, average=None, zero_division=0)

        metrics[task] = {
            "accuracy": acc, "precision_macro": precision, "recall_macro": recall,
            "f1_macro": f1_macro, "f1_weighted": f1_weighted, "per_class_f1": per_class_f1.tolist(),
            "y_true": y_true, "y_pred": y_pred, "probs": all_probs[task],
        }

    avg_loss = total_loss / max(num_batches, 1)
    overall_f1_macro = np.mean([metrics[task]["f1_macro"] for task in metrics])

    return avg_loss, metrics, overall_f1_macro


print("Task 2 training and evaluation functions defined")

Task 2 training and evaluation functions defined


In [ ]:
def run_task2_training(model_key: str, model_name: str, train_df, val_df,
                        train_config: TrainConfig, encoders: Dict[str, LabelEncoder],
                        loss_weight_config_name: str = "equal", resume: bool = True) -> Dict[str, Any]:
    print(f"\n{'='*70}\nTraining Task 2 model: {model_key} | loss_weights={loss_weight_config_name}\n{'='*70}")
    set_seed(train_config.seed)
    clear_memory()

    run_id = f"{model_key}_{loss_weight_config_name}"
    ckpt_dir = PATHS.output_root / "checkpoints" / "task2"
    ckpt_dir.mkdir(parents=True, exist_ok=True)
    ckpt_path = ckpt_dir / f"{run_id}_best.pt"
    last_ckpt_path = ckpt_dir / f"{run_id}_last.pt"

    try:
        tokenizer = AutoTokenizer.from_pretrained(model_name)
    except Exception as e:
        print(f"Failed to load tokenizer for {model_name}: {e}")
        return None

    try:
        model = MultiTaskHopeSpeechModel(model_name, TASK2_LABEL_SPACE, dropout=train_config.dropout).to(DEVICE)
    except torch.cuda.OutOfMemoryError:
        print(f"OOM while loading {model_name}. Skipping.")
        clear_memory()
        return None
    except Exception as e:
        print(f"Failed to load model {model_name}: {e}")
        return None

    train_dataset = Task2Dataset(train_df, tokenizer, train_config.max_seq_len)
    val_dataset = Task2Dataset(val_df, tokenizer, train_config.max_seq_len)

    train_collator = Task2Collator(tokenizer, is_test=False)
    val_collator = Task2Collator(tokenizer, is_test=False)

    train_loader = DataLoader(
        train_dataset, batch_size=train_config.batch_size, shuffle=True,
        collate_fn=train_collator, num_workers=NUM_WORKERS, pin_memory=PIN_MEMORY,
    )
    val_loader = DataLoader(
        val_dataset, batch_size=train_config.eval_batch_size, shuffle=False,
        collate_fn=val_collator, num_workers=NUM_WORKERS, pin_memory=PIN_MEMORY,
    )

    loss_fns = {}
    for task in ["tone", "hope_type"]:
        labels = train_df[f"{task}_id"].tolist()
        num_classes = len(encoders[task].classes_)
        class_weights = None
        if train_config.loss_type == "weighted_ce":
            class_weights = compute_class_weights_tensor(labels, num_classes, DEVICE)
        loss_fns[task] = build_loss_fn(train_config.loss_type, class_weights, train_config.focal_gamma, train_config.label_smoothing)

    task_weights = TASK2_LOSS_WEIGHT_CONFIGS[loss_weight_config_name]
    multitask_loss = MultiTaskLoss(loss_fns=loss_fns, task_weights=task_weights)

    optimizer = AdamW(model.parameters(), lr=train_config.lr, weight_decay=train_config.weight_decay)

    num_update_steps_per_epoch = math.ceil(len(train_loader) / train_config.grad_accum_steps)
    num_training_steps = num_update_steps_per_epoch * train_config.epochs
    num_warmup_steps = int(num_training_steps * train_config.warmup_ratio)
    scheduler = get_scheduler(train_config.scheduler_type, optimizer, num_warmup_steps, num_training_steps)

    scaler = GradScaler(enabled=USE_AMP)

    start_epoch = 0
    best_f1 = -1.0
    if resume and last_ckpt_path.exists():
        start_epoch, best_f1 = load_checkpoint(last_ckpt_path, model, optimizer, scheduler, scaler)
        print(f"Resumed from epoch {start_epoch} with best F1 {best_f1:.4f}")

    train_losses, val_losses = [], []
    patience_counter = 0
    history = []

    for epoch in range(start_epoch, train_config.epochs):
        epoch_start = time.time()
        train_loss = train_one_epoch_task2(
            model, train_loader, optimizer, scheduler, scaler, multitask_loss,
            train_config.grad_accum_steps, train_config.max_grad_norm, DEVICE, USE_AMP,
        )
        val_loss, val_metrics, overall_f1 = evaluate_task2(model, val_loader, multitask_loss, DEVICE, USE_AMP)

        train_losses.append(train_loss)
        val_losses.append(val_loss)
        epoch_time = time.time() - epoch_start

        print(f"Epoch {epoch+1}/{train_config.epochs} | train_loss={train_loss:.4f} | "
              f"val_loss={val_loss:.4f} | overall_f1_macro={overall_f1:.4f} | time={epoch_time:.1f}s")
        for task in ["tone", "hope_type"]:
            print(f"  {task}: f1_macro={val_metrics[task]['f1_macro']:.4f} acc={val_metrics[task]['accuracy']:.4f}")

        history.append({
            "epoch": epoch + 1, "train_loss": train_loss, "val_loss": val_loss, "overall_f1_macro": overall_f1,
            "task_f1": {task: val_metrics[task]["f1_macro"] for task in val_metrics},
        })

        save_checkpoint(last_ckpt_path, model, optimizer, scheduler, scaler, epoch + 1, best_f1, train_config)

        if overall_f1 > best_f1:
            best_f1 = overall_f1
            patience_counter = 0
            save_checkpoint(ckpt_path, model, optimizer, scheduler, scaler, epoch + 1, best_f1, train_config)
            print(f"  New best model saved. F1: {best_f1:.4f}")
        else:
            patience_counter += 1
            if patience_counter >= train_config.early_stopping_patience:
                print(f"  Early stopping at epoch {epoch+1}")
                break

    plot_learning_curves(
        train_losses, val_losses, f"Task2 {run_id} Learning Curve",
        PATHS.output_root / "plots" / f"task2_{run_id}_learning_curve.png",
    )

    log_path = PATHS.output_root / "logs" / f"task2_{run_id}_history.json"
    with open(log_path, "w") as f:
        json.dump(history, f, indent=2)

    load_checkpoint(ckpt_path, model, None, None, None)
    final_val_loss, final_val_metrics, final_f1 = evaluate_task2(model, val_loader, multitask_loss, DEVICE, USE_AMP)

    for task in ["tone", "hope_type"]:
        plot_confusion_matrix(
            final_val_metrics[task]["y_true"], final_val_metrics[task]["y_pred"],
            list(encoders[task].classes_), f"Task2 {run_id} {task} Confusion Matrix",
            PATHS.output_root / "plots" / f"task2_{run_id}_{task}_cm.png",
        )

    result = {
        "model_key": model_key, "model_name": model_name, "loss_weight_config": loss_weight_config_name,
        "best_f1_macro": best_f1,
        "final_val_metrics": {task: {k: v for k, v in final_val_metrics[task].items() if k != "probs"} for task in final_val_metrics},
        "checkpoint_path": str(ckpt_path), "tokenizer_name": model_name, "history": history,
    }

    tokenizer_save_path = ckpt_dir / f"{run_id}_tokenizer"
    tokenizer.save_pretrained(tokenizer_save_path)

    del model, optimizer, scheduler, train_loader, val_loader
    clear_memory()
    print_gpu_memory(f"after {run_id} cleanup")

    return result


print("run_task2_training defined")

run_task2_training defined


In [ ]:
task2_results = {}
task2_config = TrainConfig(
    max_seq_len=192, batch_size=8, eval_batch_size=16, grad_accum_steps=4,
    epochs=5, lr=2e-5, loss_type="weighted_ce", scheduler_type="linear",
)

for model_key in DEFAULT_TASK_MODELS:
    model_name = MODEL_REGISTRY[model_key]
    result = run_task2_training(
        model_key=model_key, model_name=model_name, train_df=task2_train, val_df=task2_val,
        train_config=task2_config, encoders=TASK2_ENCODERS, loss_weight_config_name="equal", resume=True,
    )
    if result is not None:
        task2_results[f"{model_key}_equal"] = result
    clear_memory()

best_model_key_so_far = DEFAULT_TASK_MODELS[0]
for weight_config_name in ["tone_heavy", "hope_type_heavy", "balanced_070_030"]:
    model_name = MODEL_REGISTRY[best_model_key_so_far]
    result = run_task2_training(
        model_key=best_model_key_so_far, model_name=model_name, train_df=task2_train, val_df=task2_val,
        train_config=task2_config, encoders=TASK2_ENCODERS, loss_weight_config_name=weight_config_name, resume=True,
    )
    if result is not None:
        task2_results[f"{best_model_key_so_far}_{weight_config_name}"] = result
    clear_memory()

print("\n\nTask 2 training summary:")
for key, res in task2_results.items():
    print(f"  {key}: best_f1_macro={res['best_f1_macro']:.4f}")

cross_model_keys = [k for k in task2_results.keys() if k.endswith("_equal")]
print("\nCross-model keys for ensembling:", cross_model_keys)


Training Task 2 model: xlm-roberta-base | loss_weights=equal
Resumed from epoch 5 with best F1 0.3970
[after xlm-roberta-base_equal cleanup] GPU allocated: 0.01 GB | reserved: 0.02 GB

Training Task 2 model: muril-base | loss_weights=equal
Failed to load model google/muril-base-cased: Due to a serious vulnerability issue in `torch.load`, even with `weights_only=True`, we now require users to upgrade torch to at least v2.6 in order to use the function. This version restriction does not apply when loading files with safetensors.
See the vulnerability report here https://nvd.nist.gov/vuln/detail/CVE-2025-32434

Training Task 2 model: mbert-base | loss_weights=equal
Resumed from epoch 5 with best F1 0.4516
[after mbert-base_equal cleanup] GPU allocated: 0.01 GB | reserved: 0.02 GB

Training Task 2 model: xlm-roberta-base | loss_weights=tone_heavy
Resumed from epoch 5 with best F1 0.4099
[after xlm-roberta-base_tone_heavy cleanup] GPU allocated: 0.01 GB | reserved: 0.02 GB

Training Task 2

In [ ]:
@torch.no_grad()
def get_task2_predictions(run_key: str, model_name: str, checkpoint_path: Path,
                           tokenizer_path: Path, df: pd.DataFrame, max_seq_len: int,
                           eval_batch_size: int, has_labels: bool) -> Dict[str, np.ndarray]:
    clear_memory()
    tokenizer = AutoTokenizer.from_pretrained(tokenizer_path)

    model = MultiTaskHopeSpeechModel(model_name, TASK2_LABEL_SPACE).to(DEVICE)
    load_checkpoint(checkpoint_path, model, None, None, None)
    model.eval()

    dataset = Task2Dataset(df, tokenizer, max_seq_len, is_test=not has_labels)
    collator = Task2Collator(tokenizer, is_test=not has_labels)
    loader = DataLoader(
        dataset, batch_size=eval_batch_size, shuffle=False,
        collate_fn=collator, num_workers=NUM_WORKERS, pin_memory=PIN_MEMORY,
    )

    all_probs = {"tone": [], "hope_type": []}

    for batch in loader:
        input_ids = batch["input_ids"].to(DEVICE, non_blocking=True)
        attention_mask = batch["attention_mask"].to(DEVICE, non_blocking=True)

        with autocast(device_type=DEVICE.type, enabled=USE_AMP):
            logits = model(input_ids, attention_mask)

        for task in ["tone", "hope_type"]:
            probs = F.softmax(logits[task].float(), dim=-1)
            all_probs[task].append(probs.cpu().numpy())

        del input_ids, attention_mask, logits

    for task in all_probs:
        all_probs[task] = np.concatenate(all_probs[task], axis=0)

    del model
    clear_memory()

    return all_probs


print("get_task2_predictions defined")

get_task2_predictions defined


In [ ]:
task2_val_probs = {}
task2_test_probs = {}

for run_key, result in task2_results.items():
    print(f"Collecting predictions for {run_key}")
    ckpt_dir = PATHS.output_root / "checkpoints" / "task2"
    tokenizer_path = ckpt_dir / f"{run_key}_tokenizer"
    checkpoint_path = Path(result["checkpoint_path"])

    val_probs = get_task2_predictions(
        run_key, result["model_name"], checkpoint_path, tokenizer_path,
        task2_val, task2_config.max_seq_len, task2_config.eval_batch_size, has_labels=True,
    )
    test_probs = get_task2_predictions(
        run_key, result["model_name"], checkpoint_path, tokenizer_path,
        task2_test, task2_config.max_seq_len, task2_config.eval_batch_size, has_labels=False,
    )

    task2_val_probs[run_key] = val_probs
    task2_test_probs[run_key] = test_probs
    clear_memory()

print("Collected val/test probabilities for:", list(task2_val_probs.keys()))

Collected val/test probabilities for: ['xlm-roberta-base_equal', 'mbert-base_equal', 'xlm-roberta-base_tone_heavy', 'xlm-roberta-base_hope_type_heavy', 'xlm-roberta-base_balanced_070_030']


In [ ]:
def evaluate_ensembles_task2(val_probs_by_run: Dict[str, Dict[str, np.ndarray]],
                              run_keys: List[str], val_df: pd.DataFrame,
                              encoders: Dict[str, LabelEncoder]) -> Dict[str, Any]:
    ensemble_results = {}

    for task in ["tone", "hope_type"]:
        y_true = val_df[f"{task}_id"].tolist()
        prob_list = [val_probs_by_run[rk][task] for rk in run_keys]

        strategies = {}
        hard_preds = hard_vote_ensemble(prob_list)
        strategies["hard_vote"] = {"preds": hard_preds, "f1_macro": f1_score(y_true, hard_preds, average="macro", zero_division=0)}

        soft_preds = np.argmax(soft_vote_ensemble(prob_list), axis=-1)
        strategies["soft_vote_uniform"] = {"preds": soft_preds, "f1_macro": f1_score(y_true, soft_preds, average="macro", zero_division=0)}

        rank_preds = np.argmax(rank_average_ensemble(prob_list), axis=-1)
        strategies["rank_average"] = {"preds": rank_preds, "f1_macro": f1_score(y_true, rank_preds, average="macro", zero_division=0)}

        logit_preds = np.argmax(logit_average_ensemble(prob_list), axis=-1)
        strategies["logit_average"] = {"preds": logit_preds, "f1_macro": f1_score(y_true, logit_preds, average="macro", zero_division=0)}

        opt_weights, opt_f1 = optimize_ensemble_weights(prob_list, y_true, n_trials=30)
        weighted_preds = np.argmax(soft_vote_ensemble(prob_list, opt_weights), axis=-1)
        strategies["weighted_optimized"] = {
            "preds": weighted_preds, "f1_macro": f1_score(y_true, weighted_preds, average="macro", zero_division=0),
            "weights": opt_weights,
        }

        for rk in run_keys:
            single_preds = np.argmax(val_probs_by_run[rk][task], axis=-1)
            strategies[f"single_{rk}"] = {"preds": single_preds, "f1_macro": f1_score(y_true, single_preds, average="macro", zero_division=0)}

        best_strategy = max(strategies.items(), key=lambda x: x[1]["f1_macro"])
        ensemble_results[task] = {"strategies": strategies, "best_strategy_name": best_strategy[0], "best_f1_macro": best_strategy[1]["f1_macro"]}

        print(f"\n{task} ensemble comparison:")
        for name, res in sorted(strategies.items(), key=lambda x: -x[1]["f1_macro"]):
            print(f"  {name}: f1_macro={res['f1_macro']:.4f}")
        print(f"  BEST: {best_strategy[0]} ({best_strategy[1]['f1_macro']:.4f})")

    return ensemble_results


task2_ensemble_results = evaluate_ensembles_task2(task2_val_probs, cross_model_keys, task2_val, TASK2_ENCODERS)


tone ensemble comparison:
  weighted_optimized: f1_macro=0.5195
  single_mbert-base_equal: f1_macro=0.5195
  hard_vote: f1_macro=0.5055
  rank_average: f1_macro=0.5024
  logit_average: f1_macro=0.5011
  soft_vote_uniform: f1_macro=0.4948
  single_xlm-roberta-base_equal: f1_macro=0.4718
  BEST: weighted_optimized (0.5195)

hope_type ensemble comparison:
  weighted_optimized: f1_macro=0.3864
  single_mbert-base_equal: f1_macro=0.3856
  soft_vote_uniform: f1_macro=0.3580
  rank_average: f1_macro=0.3565
  logit_average: f1_macro=0.3562
  hard_vote: f1_macro=0.3452
  single_xlm-roberta-base_equal: f1_macro=0.3219
  BEST: weighted_optimized (0.3864)


In [ ]:
best_strategy_per_task_2 = {task: task2_ensemble_results[task]["best_strategy_name"] for task in ["tone", "hope_type"]}
submission_2_strategy_t2 = {task: "soft_vote_uniform" for task in ["tone", "hope_type"]}

best_run_key = max(
    {k: v for k, v in task2_results.items() if k in cross_model_keys}.items(),
    key=lambda x: x[1]["best_f1_macro"],
)[0]
submission_3_strategy_t2 = {task: f"single_{best_run_key}" for task in ["tone", "hope_type"]}

print("submission_1 strategy per task:", best_strategy_per_task_2)
print("submission_2 strategy: uniform soft-vote")
print("submission_3 strategy: best single run =", best_run_key)

submission_1 strategy per task: {'tone': 'weighted_optimized', 'hope_type': 'weighted_optimized'}
submission_2 strategy: uniform soft-vote
submission_3 strategy: best single run = mbert-base_equal


In [ ]:
best_strategy_per_task_2 = {
    task: task2_ensemble_results[task]["best_strategy_name"] for task in ["tone", "hope_type"]
}

submission_2_strategy_t2 = {task: "soft_vote_uniform" for task in ["tone", "hope_type"]}

best_run_key = max(
    {k: v for k, v in task2_results.items() if k in cross_model_keys}.items(),
    key=lambda x: x[1]["best_f1_macro"],
)[0]
submission_3_strategy_t2 = {task: f"single_{best_run_key}" for task in ["tone", "hope_type"]}

print("Best strategy per task:", best_strategy_per_task_2)
print("Best single run:", best_run_key)

Best strategy per task: {'tone': 'weighted_optimized', 'hope_type': 'weighted_optimized'}
Best single run: mbert-base_equal


In [ ]:
def build_task2_submission(test_probs_by_run: Dict[str, Dict[str, np.ndarray]],
                            run_keys: List[str], strategy_per_task: Dict[str, str],
                            ensemble_results: Dict[str, Any], encoders: Dict[str, LabelEncoder],
                            processed_test_df: pd.DataFrame, id_column_name: str = "Id") -> pd.DataFrame:
    submission = pd.DataFrame()

    if id_column_name in processed_test_df.columns:
        submission["id"] = processed_test_df[id_column_name].values
    else:
        submission["id"] = np.arange(len(processed_test_df))

    for task in ["tone", "hope_type"]:
        strategy_name = strategy_per_task[task]
        prob_list = [test_probs_by_run[rk][task] for rk in run_keys]

        assert len(prob_list[0]) == len(submission), f"Prediction count mismatch for {task}"

        if strategy_name == "hard_vote":
            preds = hard_vote_ensemble(prob_list)
        elif strategy_name == "soft_vote_uniform":
            preds = np.argmax(soft_vote_ensemble(prob_list), axis=-1)
        elif strategy_name == "rank_average":
            preds = np.argmax(rank_average_ensemble(prob_list), axis=-1)
        elif strategy_name == "logit_average":
            preds = np.argmax(logit_average_ensemble(prob_list), axis=-1)
        elif strategy_name == "weighted_optimized":
            weights = ensemble_results[task]["strategies"]["weighted_optimized"]["weights"]
            preds = np.argmax(soft_vote_ensemble(prob_list, weights), axis=-1)
        elif strategy_name.startswith("single_"):
            rk = strategy_name.replace("single_", "")
            preds = np.argmax(test_probs_by_run[rk][task], axis=-1)
        else:
            preds = np.argmax(soft_vote_ensemble(prob_list), axis=-1)

        submission[task] = encoders[task].inverse_transform(preds)

    return submission


task2_submission_1 = build_task2_submission(
    task2_test_probs, cross_model_keys, best_strategy_per_task_2, task2_ensemble_results,
    TASK2_ENCODERS, task2_test, id_column_name="Id",
)
task2_submission_2 = build_task2_submission(
    task2_test_probs, cross_model_keys, submission_2_strategy_t2, task2_ensemble_results,
    TASK2_ENCODERS, task2_test, id_column_name="Id",
)
task2_submission_3 = build_task2_submission(
    task2_test_probs, cross_model_keys, submission_3_strategy_t2, task2_ensemble_results,
    TASK2_ENCODERS, task2_test, id_column_name="Id",
)

sub_dir = PATHS.output_root / "submissions"
task2_submission_1.to_csv(sub_dir / "task2_submission_1.csv", index=False)
task2_submission_2.to_csv(sub_dir / "task2_submission_2.csv", index=False)
task2_submission_3.to_csv(sub_dir / "task2_submission_3.csv", index=False)

print("Task 2 submissions saved. Verifying ID alignment:")
for f in ["task2_submission_1.csv", "task2_submission_2.csv", "task2_submission_3.csv"]:
    sub = pd.read_csv(sub_dir / f)
    ids_match = sub["id"].tolist() == task2_test_raw["Id"].tolist()
    print(f"  {f}: rows={len(sub)}, IDs_match_raw_order={ids_match}")
print("\nsubmission_1 preview:")
print(task2_submission_1.head())

Task 2 submissions saved. Verifying ID alignment:
  task2_submission_1.csv: rows=744, IDs_match_raw_order=True
  task2_submission_2.csv: rows=744, IDs_match_raw_order=True
  task2_submission_3.csv: rows=744, IDs_match_raw_order=True

submission_1 preview:
       id          tone        hope_type
0  HS_001   Encouraging   Inspiring Hope
1  HS_002   Encouraging  Optimistic Hope
2  HS_003   Encouraging   Realistic Hope
3  HS_004  Discouraging     Hopelessness
4  HS_005   Encouraging   Inspiring Hope


Task 3 — Kannada Hate Speech Detection (Binary)

In [ ]:
def load_task3_data():
    train_df = pd.read_csv(PATHS.task3_train)
    val_df = pd.read_csv(PATHS.task3_val)
    test_df = pd.read_csv(PATHS.task3_test)
    return train_df, val_df, test_df


task3_train_raw, task3_val_raw, task3_test_raw = load_task3_data()

print("TRAIN shape:", task3_train_raw.shape)
print("VAL shape:", task3_val_raw.shape)
print("TEST shape:", task3_test_raw.shape)
print("\nTRAIN columns:", list(task3_train_raw.columns))
print(task3_train_raw.head(3).to_string())

TRAIN shape: (3040, 4)
VAL shape: (500, 4)
TEST shape: (500, 3)

TRAIN columns: ['Unnamed: 0', 'ID', 'Comments', 'Labels']
   Unnamed: 0       ID                                     Comments  Labels
0           0  TRAIN_0                   ಸಮಾಜದಲ್ಲಿ ವಿಷ ಬಿತ್ತುವ ಹುಳು       1
1           1  TRAIN_1            Bavva ninu devaru marana kodthare       0
2           2  TRAIN_2  Ennu munde adru think madi vote madi plz 🚩🚩       0


In [ ]:
TASK3_COLUMN_MAP = {
    "text": "Comments",
    "label": "Labels",
}

TASK3_LABEL_SPACE = {"label": ["Not-hate", "Hate"]}

TASK3_RAW_LABEL_MAP = {"0": "Not-hate", "1": "Hate"}

TASK3_ENCODERS = build_label_encoders(TASK3_LABEL_SPACE)
for level, le in TASK3_ENCODERS.items():
    print(f"{level}: {list(le.classes_)}")

TASK3_HATE_ID = int(TASK3_ENCODERS["label"].transform(["Hate"])[0])
print(f"Hate class id: {TASK3_HATE_ID}")

TASK3_ID_TO_RAW_LABEL = {
    int(TASK3_ENCODERS["label"].transform(["Not-hate"])[0]): 0,
    int(TASK3_ENCODERS["label"].transform(["Hate"])[0]): 1,
}
print("Prediction ID -> submission label mapping:", TASK3_ID_TO_RAW_LABEL)

label: ['Hate', 'Not-hate']
Hate class id: 0
Prediction ID -> submission label mapping: {1: 0, 0: 1}


In [ ]:
def validate_and_prepare_task3(df: pd.DataFrame, column_map: Dict[str, str],
                                raw_label_map: Dict[str, str],
                                encoders: Dict[str, LabelEncoder]) -> pd.DataFrame:
    df = df.copy()
    text_col = column_map["text"]

    if text_col not in df.columns:
        raise KeyError(f"Text column '{text_col}' not found. Available: {list(df.columns)}")

    df = preprocess_dataframe(df, text_col, lowercase_latin_only=True, remove_emoji=False)
    df = df.rename(columns={text_col: "text"})

    label_col = column_map["label"]
    if label_col not in df.columns:
        raise KeyError(f"Label column '{label_col}' not found. Available: {list(df.columns)}")

    df[label_col] = df[label_col].astype(str).str.strip()
    df[label_col] = df[label_col].map(raw_label_map)

    invalid_mask = df[label_col].isna()
    if invalid_mask.sum() > 0:
        print(f"WARNING: {invalid_mask.sum()} rows had unmapped raw label values and were dropped")
        df = df[~invalid_mask]

    valid_classes = set(encoders["label"].classes_)
    invalid_mask2 = ~df[label_col].isin(valid_classes)
    if invalid_mask2.sum() > 0:
        df = df[~invalid_mask2]

    df["label_id"] = encoders["label"].transform(df[label_col])
    df = df.reset_index(drop=True)

    return df[["text", "label_id"]]


def validate_and_prepare_task3_test(df: pd.DataFrame, column_map: Dict[str, str],
                                     id_column_name: Optional[str] = None) -> pd.DataFrame:
    df = df.copy()
    text_col = column_map["text"]

    if text_col not in df.columns:
        raise KeyError(f"Text column '{text_col}' not found. Available: {list(df.columns)}")

    df = preprocess_dataframe_keep_all(df, text_col, lowercase_latin_only=True, remove_emoji=False)
    df = df.rename(columns={text_col: "text"})

    out_cols = ["text"]
    if id_column_name and id_column_name in df.columns:
        out_cols = [id_column_name, "text"]
    return df[out_cols].reset_index(drop=True)


task3_train = validate_and_prepare_task3(task3_train_raw, TASK3_COLUMN_MAP, TASK3_RAW_LABEL_MAP, TASK3_ENCODERS)
task3_val = validate_and_prepare_task3(task3_val_raw, TASK3_COLUMN_MAP, TASK3_RAW_LABEL_MAP, TASK3_ENCODERS)
task3_test = validate_and_prepare_task3_test(task3_test_raw, TASK3_COLUMN_MAP, id_column_name="ID")

print("Cleaned TRAIN shape:", task3_train.shape)
print("Cleaned VAL shape:", task3_val.shape)
print("TEST shape (must equal raw):", task3_test.shape, "vs raw:", task3_test_raw.shape)
assert len(task3_test) == len(task3_test_raw), "Test row count mismatch!"

print("\nLabel distribution (train):")
print(task3_train["label_id"].value_counts().sort_index())

Cleaned TRAIN shape: (3040, 2)
Cleaned VAL shape: (500, 2)
TEST shape (must equal raw): (500, 2) vs raw: (500, 3)

Label distribution (train):
label_id
0    1353
1    1687
Name: count, dtype: int64


In [ ]:
class Task3Dataset(Dataset):
    def __init__(self, df: pd.DataFrame, tokenizer, max_len: int, is_test: bool = False):
        self.texts = df["text"].tolist()
        self.is_test = is_test
        self.tokenizer = tokenizer
        self.max_len = max_len

        if not is_test:
            self.labels = df["label_id"].tolist()

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        text = self.texts[idx]
        encoding = self.tokenizer(
            text, truncation=True, max_length=self.max_len, padding=False, return_attention_mask=True,
        )
        item = {"input_ids": encoding["input_ids"], "attention_mask": encoding["attention_mask"]}
        if not self.is_test:
            item["label"] = self.labels[idx]
        return item


class Task3Collator:
    def __init__(self, tokenizer, is_test: bool = False):
        self.tokenizer = tokenizer
        self.is_test = is_test

    def __call__(self, batch):
        input_ids = [torch.tensor(x["input_ids"], dtype=torch.long) for x in batch]
        attention_mask = [torch.tensor(x["attention_mask"], dtype=torch.long) for x in batch]

        max_len = max(len(x) for x in input_ids)
        pad_id = self.tokenizer.pad_token_id if self.tokenizer.pad_token_id is not None else 0

        padded_input_ids = torch.full((len(batch), max_len), pad_id, dtype=torch.long)
        padded_attention_mask = torch.zeros((len(batch), max_len), dtype=torch.long)

        for i, (ids, mask) in enumerate(zip(input_ids, attention_mask)):
            length = len(ids)
            padded_input_ids[i, :length] = ids
            padded_attention_mask[i, :length] = mask

        output = {"input_ids": padded_input_ids, "attention_mask": padded_attention_mask}

        if not self.is_test:
            output["label"] = torch.tensor([x["label"] for x in batch], dtype=torch.long)

        return output


print("Task3Dataset and Task3Collator defined")

Task3Dataset and Task3Collator defined


In [ ]:
class BinaryHateSpeechModel(nn.Module):
    def __init__(self, model_name: str, num_classes: int = 2,
                 dropout: float = 0.1, use_gradient_checkpointing: bool = True):
        super().__init__()
        self.config = AutoConfig.from_pretrained(model_name)
        self.encoder = AutoModel.from_pretrained(model_name)

        if use_gradient_checkpointing and hasattr(self.encoder, "gradient_checkpointing_enable"):
            self.encoder.gradient_checkpointing_enable()

        hidden_size = self.config.hidden_size
        self.dropout = nn.Dropout(dropout)
        self.classifier = nn.Sequential(
            nn.Linear(hidden_size, hidden_size // 2), nn.GELU(), nn.Dropout(dropout),
            nn.Linear(hidden_size // 2, num_classes),
        )

    def mean_pooling(self, token_embeddings, attention_mask):
        mask_expanded = attention_mask.unsqueeze(-1).expand(token_embeddings.size()).float()
        sum_embeddings = torch.sum(token_embeddings * mask_expanded, dim=1)
        sum_mask = torch.clamp(mask_expanded.sum(dim=1), min=1e-9)
        return sum_embeddings / sum_mask

    def forward(self, input_ids, attention_mask):
        outputs = self.encoder(input_ids=input_ids, attention_mask=attention_mask)
        last_hidden = outputs.last_hidden_state
        pooled = self.mean_pooling(last_hidden, attention_mask)
        pooled = self.dropout(pooled)
        return self.classifier(pooled)


print("BinaryHateSpeechModel defined")

BinaryHateSpeechModel defined


In [ ]:
def train_one_epoch_task3(model, loader, optimizer, scheduler, scaler, loss_fn,
                           grad_accum_steps, max_grad_norm, device, use_amp):
    model.train()
    total_loss = 0.0
    num_batches = 0
    optimizer.zero_grad(set_to_none=True)

    for step, batch in enumerate(loader):
        try:
            input_ids = batch["input_ids"].to(device, non_blocking=True)
            attention_mask = batch["attention_mask"].to(device, non_blocking=True)
            labels = batch["label"].to(device, non_blocking=True)

            with autocast(device_type=device.type, enabled=use_amp):
                logits = model(input_ids, attention_mask)
                loss = loss_fn(logits, labels)
                loss = loss / grad_accum_steps

            if use_amp:
                scaler.scale(loss).backward()
            else:
                loss.backward()

            if (step + 1) % grad_accum_steps == 0:
                if use_amp:
                    scaler.unscale_(optimizer)
                    torch.nn.utils.clip_grad_norm_(model.parameters(), max_grad_norm)
                    scaler.step(optimizer)
                    scaler.update()
                else:
                    torch.nn.utils.clip_grad_norm_(model.parameters(), max_grad_norm)
                    optimizer.step()
                optimizer.zero_grad(set_to_none=True)
                if scheduler is not None:
                    scheduler.step()

            total_loss += loss.item() * grad_accum_steps
            num_batches += 1

            del input_ids, attention_mask, labels, logits, loss

        except torch.cuda.OutOfMemoryError:
            print(f"OOM at step {step}. Clearing cache and skipping batch.")
            optimizer.zero_grad(set_to_none=True)
            clear_memory()
            continue

    if (num_batches % grad_accum_steps) != 0:
        if use_amp:
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_grad_norm)
            scaler.step(optimizer)
            scaler.update()
        else:
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_grad_norm)
            optimizer.step()
        optimizer.zero_grad(set_to_none=True)

    clear_memory()
    return total_loss / max(num_batches, 1)


@torch.no_grad()
def evaluate_task3(model, loader, loss_fn, device, use_amp):
    model.eval()
    total_loss = 0.0
    num_batches = 0

    all_preds, all_targets, all_probs = [], [], []

    for batch in loader:
        input_ids = batch["input_ids"].to(device, non_blocking=True)
        attention_mask = batch["attention_mask"].to(device, non_blocking=True)
        labels = batch["label"].to(device, non_blocking=True)

        with autocast(device_type=device.type, enabled=use_amp):
            logits = model(input_ids, attention_mask)
            loss = loss_fn(logits, labels)

        total_loss += loss.item()
        num_batches += 1

        probs = F.softmax(logits.float(), dim=-1)
        preds = torch.argmax(probs, dim=-1)

        all_preds.extend(preds.cpu().numpy().tolist())
        all_targets.extend(labels.cpu().numpy().tolist())
        all_probs.extend(probs.cpu().numpy().tolist())

        del input_ids, attention_mask, labels, logits

    clear_memory()

    acc = accuracy_score(all_targets, all_preds)
    precision, recall, f1_macro, _ = precision_recall_fscore_support(all_targets, all_preds, average="macro", zero_division=0)
    _, _, f1_weighted, _ = precision_recall_fscore_support(all_targets, all_preds, average="weighted", zero_division=0)
    per_class_f1 = f1_score(all_targets, all_preds, average=None, zero_division=0)

    try:
        hate_probs = [p[TASK3_HATE_ID] for p in all_probs]
        auc = roc_auc_score(all_targets, hate_probs)
    except ValueError:
        auc = float("nan")

    metrics = {
        "accuracy": acc, "precision_macro": precision, "recall_macro": recall,
        "f1_macro": f1_macro, "f1_weighted": f1_weighted, "roc_auc": auc,
        "per_class_f1": per_class_f1.tolist(), "y_true": all_targets, "y_pred": all_preds, "probs": all_probs,
    }

    avg_loss = total_loss / max(num_batches, 1)
    return avg_loss, metrics


print("Task 3 training and evaluation functions defined")

Task 3 training and evaluation functions defined


In [ ]:
def run_task3_training(model_key: str, model_name: str, train_df, val_df,
                        train_config: TrainConfig, encoders: Dict[str, LabelEncoder],
                        resume: bool = True) -> Dict[str, Any]:
    print(f"\n{'='*70}\nTraining Task 3 model: {model_key} ({model_name})\n{'='*70}")
    set_seed(train_config.seed)
    clear_memory()

    ckpt_dir = PATHS.output_root / "checkpoints" / "task3"
    ckpt_dir.mkdir(parents=True, exist_ok=True)
    ckpt_path = ckpt_dir / f"{model_key}_best.pt"
    last_ckpt_path = ckpt_dir / f"{model_key}_last.pt"

    try:
        tokenizer = AutoTokenizer.from_pretrained(model_name)
    except Exception as e:
        print(f"Failed to load tokenizer for {model_name}: {e}")
        return None

    try:
        model = BinaryHateSpeechModel(model_name, num_classes=2, dropout=train_config.dropout).to(DEVICE)
    except torch.cuda.OutOfMemoryError:
        print(f"OOM while loading {model_name}. Skipping.")
        clear_memory()
        return None
    except Exception as e:
        print(f"Failed to load model {model_name}: {e}")
        return None

    train_dataset = Task3Dataset(train_df, tokenizer, train_config.max_seq_len)
    val_dataset = Task3Dataset(val_df, tokenizer, train_config.max_seq_len)

    train_collator = Task3Collator(tokenizer, is_test=False)
    val_collator = Task3Collator(tokenizer, is_test=False)

    train_loader = DataLoader(
        train_dataset, batch_size=train_config.batch_size, shuffle=True,
        collate_fn=train_collator, num_workers=NUM_WORKERS, pin_memory=PIN_MEMORY,
    )
    val_loader = DataLoader(
        val_dataset, batch_size=train_config.eval_batch_size, shuffle=False,
        collate_fn=val_collator, num_workers=NUM_WORKERS, pin_memory=PIN_MEMORY,
    )

    labels_list = train_df["label_id"].tolist()
    class_weights = None
    if train_config.loss_type == "weighted_ce":
        class_weights = compute_class_weights_tensor(labels_list, 2, DEVICE)
    loss_fn = build_loss_fn(train_config.loss_type, class_weights, train_config.focal_gamma, train_config.label_smoothing)

    optimizer = AdamW(model.parameters(), lr=train_config.lr, weight_decay=train_config.weight_decay)

    num_update_steps_per_epoch = math.ceil(len(train_loader) / train_config.grad_accum_steps)
    num_training_steps = num_update_steps_per_epoch * train_config.epochs
    num_warmup_steps = int(num_training_steps * train_config.warmup_ratio)
    scheduler = get_scheduler(train_config.scheduler_type, optimizer, num_warmup_steps, num_training_steps)

    scaler = GradScaler(enabled=USE_AMP)

    start_epoch = 0
    best_f1 = -1.0
    if resume and last_ckpt_path.exists():
        start_epoch, best_f1 = load_checkpoint(last_ckpt_path, model, optimizer, scheduler, scaler)
        print(f"Resumed from epoch {start_epoch} with best F1 {best_f1:.4f}")

    train_losses, val_losses = [], []
    patience_counter = 0
    history = []

    for epoch in range(start_epoch, train_config.epochs):
        epoch_start = time.time()
        train_loss = train_one_epoch_task3(
            model, train_loader, optimizer, scheduler, scaler, loss_fn,
            train_config.grad_accum_steps, train_config.max_grad_norm, DEVICE, USE_AMP,
        )
        val_loss, val_metrics = evaluate_task3(model, val_loader, loss_fn, DEVICE, USE_AMP)

        train_losses.append(train_loss)
        val_losses.append(val_loss)
        epoch_time = time.time() - epoch_start

        print(f"Epoch {epoch+1}/{train_config.epochs} | train_loss={train_loss:.4f} | "
              f"val_loss={val_loss:.4f} | f1_macro={val_metrics['f1_macro']:.4f} | "
              f"acc={val_metrics['accuracy']:.4f} | auc={val_metrics['roc_auc']:.4f} | time={epoch_time:.1f}s")

        history.append({
            "epoch": epoch + 1, "train_loss": train_loss, "val_loss": val_loss,
            "f1_macro": val_metrics["f1_macro"], "accuracy": val_metrics["accuracy"], "roc_auc": val_metrics["roc_auc"],
        })

        save_checkpoint(last_ckpt_path, model, optimizer, scheduler, scaler, epoch + 1, best_f1, train_config)

        if val_metrics["f1_macro"] > best_f1:
            best_f1 = val_metrics["f1_macro"]
            patience_counter = 0
            save_checkpoint(ckpt_path, model, optimizer, scheduler, scaler, epoch + 1, best_f1, train_config)
            print(f"  New best model saved. F1: {best_f1:.4f}")
        else:
            patience_counter += 1
            if patience_counter >= train_config.early_stopping_patience:
                print(f"  Early stopping at epoch {epoch+1}")
                break

    plot_learning_curves(
        train_losses, val_losses, f"Task3 {model_key} Learning Curve",
        PATHS.output_root / "plots" / f"task3_{model_key}_learning_curve.png",
    )

    log_path = PATHS.output_root / "logs" / f"task3_{model_key}_history.json"
    with open(log_path, "w") as f:
        json.dump(history, f, indent=2)

    load_checkpoint(ckpt_path, model, None, None, None)
    final_val_loss, final_val_metrics = evaluate_task3(model, val_loader, loss_fn, DEVICE, USE_AMP)

    plot_confusion_matrix(
        final_val_metrics["y_true"], final_val_metrics["y_pred"],
        list(encoders["label"].classes_), f"Task3 {model_key} Confusion Matrix",
        PATHS.output_root / "plots" / f"task3_{model_key}_cm.png",
    )

    print("\nClassification report:")
    print(classification_report(
        final_val_metrics["y_true"], final_val_metrics["y_pred"],
        target_names=list(encoders["label"].classes_), zero_division=0,
    ))

    result = {
        "model_key": model_key, "model_name": model_name, "best_f1_macro": best_f1,
        "final_val_metrics": {k: v for k, v in final_val_metrics.items() if k != "probs"},
        "val_probs": final_val_metrics["probs"],
        "checkpoint_path": str(ckpt_path), "tokenizer_name": model_name, "history": history,
    }

    tokenizer_save_path = ckpt_dir / f"{model_key}_tokenizer"
    tokenizer.save_pretrained(tokenizer_save_path)

    del model, optimizer, scheduler, train_loader, val_loader
    clear_memory()
    print_gpu_memory(f"after {model_key} cleanup")

    return result


print("run_task3_training defined")

run_task3_training defined


In [ ]:
task3_results = {}
task3_config = TrainConfig(
    max_seq_len=192, batch_size=8, eval_batch_size=16, grad_accum_steps=4,
    epochs=5, lr=2e-5, loss_type="weighted_ce", scheduler_type="linear",
)

for model_key in DEFAULT_TASK_MODELS:
    model_name = MODEL_REGISTRY[model_key]
    result = run_task3_training(
        model_key=model_key, model_name=model_name, train_df=task3_train, val_df=task3_val,
        train_config=task3_config, encoders=TASK3_ENCODERS, resume=True,
    )
    if result is not None:
        task3_results[model_key] = result
    clear_memory()

print("\n\nTask 3 training summary:")
for key, res in task3_results.items():
    print(f"  {key}: best_f1_macro={res['best_f1_macro']:.4f}")


Training Task 3 model: xlm-roberta-base (xlm-roberta-base)
Resumed from epoch 5 with best F1 0.6918

Classification report:
              precision    recall  f1-score   support

        Hate       0.65      0.68      0.67       212
    Not-hate       0.76      0.73      0.75       288

    accuracy                           0.71       500
   macro avg       0.71      0.71      0.71       500
weighted avg       0.71      0.71      0.71       500

[after xlm-roberta-base cleanup] GPU allocated: 0.01 GB | reserved: 0.02 GB

Training Task 3 model: muril-base (google/muril-base-cased)
Failed to load model google/muril-base-cased: Due to a serious vulnerability issue in `torch.load`, even with `weights_only=True`, we now require users to upgrade torch to at least v2.6 in order to use the function. This version restriction does not apply when loading files with safetensors.
See the vulnerability report here https://nvd.nist.gov/vuln/detail/CVE-2025-32434

Training Task 3 model: mbert-base (

In [ ]:
task3_thresholds = {}
for model_key, result in task3_results.items():
    y_true = result["final_val_metrics"]["y_true"]
    hate_probs = [p[TASK3_HATE_ID] for p in result["val_probs"]]
    best_t, best_f1 = optimize_threshold_for_macro_f1(y_true, hate_probs, TASK3_HATE_ID)
    task3_thresholds[model_key] = best_t
    print(f"{model_key}: optimal threshold={best_t:.2f}, f1_macro at threshold={best_f1:.4f}, "
          f"default (argmax) f1_macro={result['best_f1_macro']:.4f}")

xlm-roberta-base: optimal threshold=0.34, f1_macro at threshold=0.7198, default (argmax) f1_macro=0.6918
mbert-base: optimal threshold=0.40, f1_macro at threshold=0.7447, default (argmax) f1_macro=0.7343


In [ ]:
@torch.no_grad()
def get_task3_predictions(model_key: str, model_name: str, checkpoint_path: Path,
                           tokenizer_path: Path, df: pd.DataFrame, max_seq_len: int,
                           eval_batch_size: int, has_labels: bool) -> np.ndarray:
    clear_memory()
    tokenizer = AutoTokenizer.from_pretrained(tokenizer_path)

    model = BinaryHateSpeechModel(model_name, num_classes=2).to(DEVICE)
    load_checkpoint(checkpoint_path, model, None, None, None)
    model.eval()

    dataset = Task3Dataset(df, tokenizer, max_seq_len, is_test=not has_labels)
    collator = Task3Collator(tokenizer, is_test=not has_labels)
    loader = DataLoader(
        dataset, batch_size=eval_batch_size, shuffle=False,
        collate_fn=collator, num_workers=NUM_WORKERS, pin_memory=PIN_MEMORY,
    )

    all_probs = []

    for batch in loader:
        input_ids = batch["input_ids"].to(DEVICE, non_blocking=True)
        attention_mask = batch["attention_mask"].to(DEVICE, non_blocking=True)

        with autocast(device_type=DEVICE.type, enabled=USE_AMP):
            logits = model(input_ids, attention_mask)

        probs = F.softmax(logits.float(), dim=-1)
        all_probs.append(probs.cpu().numpy())

        del input_ids, attention_mask, logits

    all_probs = np.concatenate(all_probs, axis=0)

    del model
    clear_memory()

    return all_probs


print("get_task3_predictions defined")

get_task3_predictions defined


In [ ]:
@torch.no_grad()
def get_task3_predictions(model_key: str, model_name: str, checkpoint_path: Path,
                           tokenizer_path: Path, df: pd.DataFrame, max_seq_len: int,
                           eval_batch_size: int, has_labels: bool) -> np.ndarray:
    clear_memory()
    tokenizer = AutoTokenizer.from_pretrained(tokenizer_path)

    model = BinaryHateSpeechModel(model_name, num_classes=2).to(DEVICE)
    load_checkpoint(checkpoint_path, model, None, None, None)
    model.eval()

    dataset = Task3Dataset(df, tokenizer, max_seq_len, is_test=not has_labels)
    collator = Task3Collator(tokenizer, is_test=not has_labels)
    loader = DataLoader(
        dataset, batch_size=eval_batch_size, shuffle=False,
        collate_fn=collator, num_workers=NUM_WORKERS, pin_memory=PIN_MEMORY,
    )

    all_probs = []

    for batch in loader:
        input_ids = batch["input_ids"].to(DEVICE, non_blocking=True)
        attention_mask = batch["attention_mask"].to(DEVICE, non_blocking=True)

        with autocast(device_type=DEVICE.type, enabled=USE_AMP):
            logits = model(input_ids, attention_mask)

        probs = F.softmax(logits.float(), dim=-1)
        all_probs.append(probs.cpu().numpy())

        del input_ids, attention_mask, logits

    all_probs = np.concatenate(all_probs, axis=0)

    del model
    clear_memory()

    return all_probs


print("get_task3_predictions function defined")

get_task3_predictions function defined


In [ ]:
task3_test_probs = {}

for model_key, result in task3_results.items():
    print(f"Collecting test predictions for {model_key}")
    ckpt_dir = PATHS.output_root / "checkpoints" / "task3"
    tokenizer_path = ckpt_dir / f"{model_key}_tokenizer"
    checkpoint_path = Path(result["checkpoint_path"])

    test_probs = get_task3_predictions(
        model_key, result["model_name"], checkpoint_path, tokenizer_path,
        task3_test, task3_config.max_seq_len, task3_config.eval_batch_size, has_labels=False,
    )
    task3_test_probs[model_key] = test_probs
    clear_memory()

print("Collected test probabilities for:", list(task3_test_probs.keys()))

Collected test probabilities for: ['xlm-roberta-base', 'mbert-base']


In [ ]:
task3_val_prob_list = [np.array(task3_results[mk]["val_probs"]) for mk in task3_results.keys()]
task3_model_keys = list(task3_results.keys())
task3_y_true_val = task3_results[task3_model_keys[0]]["final_val_metrics"]["y_true"]

def evaluate_ensembles_task3(prob_list: List[np.ndarray], y_true: List[int],
                              model_keys: List[str], hate_class_id: int) -> Dict[str, Any]:
    strategies = {}

    hard_preds = hard_vote_ensemble(prob_list)
    strategies["hard_vote"] = {"preds": hard_preds, "f1_macro": f1_score(y_true, hard_preds, average="macro", zero_division=0)}

    soft_probs = soft_vote_ensemble(prob_list)
    soft_preds = np.argmax(soft_probs, axis=-1)
    strategies["soft_vote_uniform"] = {"preds": soft_preds, "f1_macro": f1_score(y_true, soft_preds, average="macro", zero_division=0)}

    rank_probs = rank_average_ensemble(prob_list)
    rank_preds = np.argmax(rank_probs, axis=-1)
    strategies["rank_average"] = {"preds": rank_preds, "f1_macro": f1_score(y_true, rank_preds, average="macro", zero_division=0)}

    logit_probs = logit_average_ensemble(prob_list)
    logit_preds = np.argmax(logit_probs, axis=-1)
    strategies["logit_average"] = {"preds": logit_preds, "f1_macro": f1_score(y_true, logit_preds, average="macro", zero_division=0)}

    opt_weights, opt_f1 = optimize_ensemble_weights(prob_list, y_true, n_trials=30)
    weighted_probs = soft_vote_ensemble(prob_list, opt_weights)
    weighted_preds = np.argmax(weighted_probs, axis=-1)
    strategies["weighted_optimized"] = {
        "preds": weighted_preds, "f1_macro": f1_score(y_true, weighted_preds, average="macro", zero_division=0),
        "weights": opt_weights,
    }

    soft_hate_probs = soft_probs[:, hate_class_id].tolist()
    ens_threshold, ens_threshold_f1 = optimize_threshold_for_macro_f1(y_true, soft_hate_probs, hate_class_id)
    ens_threshold_preds = np.array([hate_class_id if p >= ens_threshold else (1 - hate_class_id) for p in soft_hate_probs])
    strategies["soft_vote_thresholded"] = {"preds": ens_threshold_preds, "f1_macro": ens_threshold_f1, "threshold": ens_threshold}

    for mk, probs in zip(model_keys, prob_list):
        single_preds = np.argmax(probs, axis=-1)
        strategies[f"single_{mk}"] = {"preds": single_preds, "f1_macro": f1_score(y_true, single_preds, average="macro", zero_division=0)}
        hate_probs_single = probs[:, hate_class_id].tolist()
        t, tf1 = optimize_threshold_for_macro_f1(y_true, hate_probs_single, hate_class_id)
        strategies[f"single_{mk}_thresholded"] = {
            "preds": np.array([hate_class_id if p >= t else (1 - hate_class_id) for p in hate_probs_single]),
            "f1_macro": tf1, "threshold": t,
        }

    best_strategy = max(strategies.items(), key=lambda x: x[1]["f1_macro"])

    print("Task 3 ensemble comparison:")
    for name, res in sorted(strategies.items(), key=lambda x: -x[1]["f1_macro"]):
        print(f"  {name}: f1_macro={res['f1_macro']:.4f}")
    print(f"  BEST: {best_strategy[0]} ({best_strategy[1]['f1_macro']:.4f})")

    return {"strategies": strategies, "best_strategy_name": best_strategy[0], "best_f1_macro": best_strategy[1]["f1_macro"]}


task3_ensemble_results = evaluate_ensembles_task3(task3_val_prob_list, task3_y_true_val, task3_model_keys, TASK3_HATE_ID)

Task 3 ensemble comparison:
  soft_vote_thresholded: f1_macro=0.7657
  weighted_optimized: f1_macro=0.7574
  soft_vote_uniform: f1_macro=0.7536
  logit_average: f1_macro=0.7536
  single_mbert-base_thresholded: f1_macro=0.7447
  single_mbert-base: f1_macro=0.7343
  hard_vote: f1_macro=0.7340
  rank_average: f1_macro=0.7340
  single_xlm-roberta-base_thresholded: f1_macro=0.7198
  single_xlm-roberta-base: f1_macro=0.7069
  BEST: soft_vote_thresholded (0.7657)


In [ ]:
submission_1_strategy_t3 = task3_ensemble_results["best_strategy_name"]
submission_2_strategy_t3 = "soft_vote_thresholded"

best_single_model_t3 = max(task3_results.items(), key=lambda x: x[1]["best_f1_macro"])[0]
submission_3_strategy_t3 = f"single_{best_single_model_t3}_thresholded"

print("submission_1 strategy:", submission_1_strategy_t3)
print("submission_2 strategy:", submission_2_strategy_t3)
print("submission_3 strategy:", submission_3_strategy_t3)

submission_1 strategy: soft_vote_thresholded
submission_2 strategy: soft_vote_thresholded
submission_3 strategy: single_mbert-base_thresholded


In [ ]:
def build_task3_submission(test_probs_by_model: Dict[str, np.ndarray], model_keys: List[str],
                            strategy_name: str, ensemble_results: Dict[str, Any],
                            hate_class_id: int, processed_test_df: pd.DataFrame,
                            id_column_name: str = "ID") -> pd.DataFrame:
    submission = pd.DataFrame()

    if id_column_name in processed_test_df.columns:
        submission["id"] = processed_test_df[id_column_name].values
    else:
        submission["id"] = np.arange(len(processed_test_df))

    prob_list = [test_probs_by_model[mk] for mk in model_keys]
    strategy_info = ensemble_results["strategies"][strategy_name]

    assert len(prob_list[0]) == len(submission), "Prediction count mismatch"

    if strategy_name == "hard_vote":
        preds = hard_vote_ensemble(prob_list)
    elif strategy_name == "soft_vote_uniform":
        preds = np.argmax(soft_vote_ensemble(prob_list), axis=-1)
    elif strategy_name == "rank_average":
        preds = np.argmax(rank_average_ensemble(prob_list), axis=-1)
    elif strategy_name == "logit_average":
        preds = np.argmax(logit_average_ensemble(prob_list), axis=-1)
    elif strategy_name == "weighted_optimized":
        weights = strategy_info["weights"]
        preds = np.argmax(soft_vote_ensemble(prob_list, weights), axis=-1)
    elif strategy_name == "soft_vote_thresholded":
        threshold = strategy_info["threshold"]
        soft_probs = soft_vote_ensemble(prob_list)
        hate_probs = soft_probs[:, hate_class_id]
        preds = np.array([hate_class_id if p >= threshold else (1 - hate_class_id) for p in hate_probs])
    elif strategy_name.startswith("single_") and strategy_name.endswith("_thresholded"):
        mk = strategy_name.replace("single_", "").replace("_thresholded", "")
        threshold = strategy_info["threshold"]
        hate_probs = test_probs_by_model[mk][:, hate_class_id]
        preds = np.array([hate_class_id if p >= threshold else (1 - hate_class_id) for p in hate_probs])
    elif strategy_name.startswith("single_"):
        mk = strategy_name.replace("single_", "")
        preds = np.argmax(test_probs_by_model[mk], axis=-1)
    else:
        preds = np.argmax(soft_vote_ensemble(prob_list), axis=-1)

    submission["label"] = [TASK3_ID_TO_RAW_LABEL[int(p)] for p in preds]
    return submission


task3_submission_1 = build_task3_submission(
    task3_test_probs, task3_model_keys, submission_1_strategy_t3, task3_ensemble_results,
    TASK3_HATE_ID, task3_test, id_column_name="ID",
)
task3_submission_2 = build_task3_submission(
    task3_test_probs, task3_model_keys, submission_2_strategy_t3, task3_ensemble_results,
    TASK3_HATE_ID, task3_test, id_column_name="ID",
)
task3_submission_3 = build_task3_submission(
    task3_test_probs, task3_model_keys, submission_3_strategy_t3, task3_ensemble_results,
    TASK3_HATE_ID, task3_test, id_column_name="ID",
)

sub_dir = PATHS.output_root / "submissions"
task3_submission_1.to_csv(sub_dir / "task3_submission_1.csv", index=False)
task3_submission_2.to_csv(sub_dir / "task3_submission_2.csv", index=False)
task3_submission_3.to_csv(sub_dir / "task3_submission_3.csv", index=False)

print("Task 3 submissions saved:")
for f in ["task3_submission_1.csv", "task3_submission_2.csv", "task3_submission_3.csv"]:
    sub = pd.read_csv(sub_dir / f)
    print(f"\n{f}:")
    print(sub.head())
    print("label dtype:", sub["label"].dtype)
    ids_match = sub["id"].tolist() == task3_test_raw["ID"].tolist()
    print("IDs match raw order:", ids_match)

Task 3 submissions saved:

task3_submission_1.csv:
       id  label
0  TEST_0      0
1  TEST_1      1
2  TEST_2      1
3  TEST_3      1
4  TEST_4      1
label dtype: int64
IDs match raw order: True

task3_submission_2.csv:
       id  label
0  TEST_0      0
1  TEST_1      1
2  TEST_2      1
3  TEST_3      1
4  TEST_4      1
label dtype: int64
IDs match raw order: True

task3_submission_3.csv:
       id  label
0  TEST_0      0
1  TEST_1      1
2  TEST_2      1
3  TEST_3      1
4  TEST_4      1
label dtype: int64
IDs match raw order: True


Error Analysis

In [ ]:
task1_test_labeled_raw = pd.read_csv(PATHS.task1_test_labeled)

print("Labeled test shape:", task1_test_labeled_raw.shape)
print("Columns:", list(task1_test_labeled_raw.columns))
print(task1_test_labeled_raw.head(3).to_string())

Labeled test shape: (532, 7)
Columns: ['Input_ID', 'Input_Text', 'Level1_Class', 'Level2_Target', 'Level3_Category', 'Level4_Subcategory', 'Level5_Bias']
   Input_ID                                                                                                                                                                 Input_Text Level1_Class     Level2_Target Level3_Category Level4_Subcategory Level5_Bias
0      2604                                                                      நாசமா போன நாதாரிகளே..... ஏன்டா தங்கம் தங்கம்னு சாவுரிங்க???? சீதனம் வாங்கி தின்னுட்டு சாவுங்கடா......         HATE  Hate against Men        IMPLICIT                Nil         Nil
1      4803                                                                                                                நீ பேசாம இரு திவ்யா நான் சிரிச்சுட போறேன்.....  Super....po     NON-HATE               Nil             Nil                Nil         Nil
2      3512  வாழ்க்கை என்று வரும்போது பணம் கௌரவம் முக்கிய பங்கா

In [ ]:
def validate_and_prepare_task1_labeled_test(df: pd.DataFrame, column_map: Dict[str, str],
                                              encoders: Dict[str, LabelEncoder],
                                              id_column_name: str) -> pd.DataFrame:
    df = df.copy()
    text_col = column_map["text"]

    if text_col not in df.columns:
        raise KeyError(f"Text column '{text_col}' not found. Available: {list(df.columns)}")

    df = preprocess_dataframe_keep_all(df, text_col)
    df = df.rename(columns={text_col: "text"})

    normalize_fns = {
        "level1": normalize_level1, "level2": normalize_level2,
        "level3": normalize_level3, "level4": normalize_level4, "level5": normalize_level5,
    }

    for level in ["level1", "level2", "level3", "level4", "level5"]:
        col = column_map[level]
        if col not in df.columns:
            raise KeyError(f"Label column '{col}' for {level} not found. Available: {list(df.columns)}")
        df[col] = df[col].astype(str).apply(normalize_fns[level])

    before_drop = len(df)
    df = df[df[column_map["level1"]].notna()].reset_index(drop=True)
    dropped_l1 = before_drop - len(df)
    if dropped_l1 > 0:
        print(f"WARNING: Dropped {dropped_l1} labeled-test rows with unresolved Level1 label.")

    keep_cols = ["text", id_column_name] if id_column_name in df.columns else ["text"]
    for level in ["level1", "level2", "level3", "level4", "level5"]:
        col = column_map[level]
        valid_classes = set(encoders[level].classes_)
        invalid_mask = ~df[col].isin(valid_classes)
        if invalid_mask.sum() > 0:
            print(f"WARNING: {invalid_mask.sum()} labeled-test rows in {level} outside label space: "
                  f"{sorted(df.loc[invalid_mask, col].unique())[:10]}")
            df = df[~invalid_mask]
        df[f"{level}_id"] = encoders[level].transform(df[col])
        keep_cols.append(f"{level}_id")

    df = df.reset_index(drop=True)
    return df[keep_cols]


task1_test_labeled = validate_and_prepare_task1_labeled_test(
    task1_test_labeled_raw, TASK1_COLUMN_MAP, TASK1_ENCODERS, id_column_name="Input_ID",
)

print("Labeled test (cleaned) shape:", task1_test_labeled.shape)
for level in ["level1", "level2", "level3", "level4", "level5"]:
    print(f"\n{level} distribution (labeled test):")
    print(task1_test_labeled[f"{level}_id"].value_counts().sort_index())

Labeled test (cleaned) shape: (531, 7)

level1 distribution (labeled test):
level1_id
0    279
1    252
Name: count, dtype: int64

level2 distribution (labeled test):
level2_id
0     20
1     89
2    120
3    254
4     48
Name: count, dtype: int64

level3 distribution (labeled test):
level3_id
0    122
1    157
2    252
Name: count, dtype: int64

level4 distribution (labeled test):
level4_id
0       6
1      28
2      13
4       3
5       5
6       1
7       9
8       8
9     307
11      4
12     39
13      4
14     18
15      2
16     78
17      6
Name: count, dtype: int64

level5 distribution (labeled test):
level5_id
0      1
1     12
2     23
3      6
4    479
6     10
Name: count, dtype: int64


In [ ]:
task1_labeled_test_probs = {}

for model_key, result in task1_results.items():
    print(f"Collecting labeled-test predictions for {model_key}")
    ckpt_dir = PATHS.output_root / "checkpoints" / "task1"
    tokenizer_path = ckpt_dir / f"{model_key}_tokenizer"
    checkpoint_path = Path(result["checkpoint_path"])

    probs = get_task1_predictions(
        model_key, result["model_name"], checkpoint_path, tokenizer_path,
        task1_test_labeled, task1_config.max_seq_len, task1_config.eval_batch_size, has_labels=False,
    )
    task1_labeled_test_probs[model_key] = probs
    clear_memory()

print("Collected labeled-test probabilities for:", list(task1_labeled_test_probs.keys()))

Collected labeled-test probabilities for: ['xlm-roberta-base', 'mbert-base']


In [ ]:
def apply_ensemble_strategy_task1(prob_list: List[np.ndarray], strategy_name: str,
                                   ensemble_results_for_level: Dict[str, Any],
                                   model_keys_ordered: List[str]) -> np.ndarray:
    if strategy_name == "hard_vote":
        return hard_vote_ensemble(prob_list)
    elif strategy_name == "soft_vote_uniform":
        return np.argmax(soft_vote_ensemble(prob_list), axis=-1)
    elif strategy_name == "rank_average":
        return np.argmax(rank_average_ensemble(prob_list), axis=-1)
    elif strategy_name == "logit_average":
        return np.argmax(logit_average_ensemble(prob_list), axis=-1)
    elif strategy_name == "weighted_optimized":
        weights = ensemble_results_for_level["strategies"]["weighted_optimized"]["weights"]
        return np.argmax(soft_vote_ensemble(prob_list, weights), axis=-1)
    elif strategy_name.startswith("single_"):
        mk = strategy_name.replace("single_", "")
        idx = model_keys_ordered.index(mk)
        return np.argmax(prob_list[idx], axis=-1)
    else:
        return np.argmax(soft_vote_ensemble(prob_list), axis=-1)


task1_model_keys_ordered = list(task1_val_probs.keys())

task1_run1_val_preds_raw = {}
task1_run2_val_preds_raw = {}
task1_run1_test_preds_raw = {}
task1_run2_test_preds_raw = {}

for level in ["level1", "level2", "level3", "level4", "level5"]:
    val_prob_list = [task1_val_probs[mk][level] for mk in task1_model_keys_ordered]
    test_prob_list = [task1_labeled_test_probs[mk][level] for mk in task1_model_keys_ordered]

    strategy_1 = best_strategy_per_level[level]
    strategy_2 = "soft_vote_uniform"

    task1_run1_val_preds_raw[level] = apply_ensemble_strategy_task1(
        val_prob_list, strategy_1, task1_ensemble_results[level], task1_model_keys_ordered,
    )
    task1_run2_val_preds_raw[level] = apply_ensemble_strategy_task1(
        val_prob_list, strategy_2, task1_ensemble_results[level], task1_model_keys_ordered,
    )
    task1_run1_test_preds_raw[level] = apply_ensemble_strategy_task1(
        test_prob_list, strategy_1, task1_ensemble_results[level], task1_model_keys_ordered,
    )
    task1_run2_test_preds_raw[level] = apply_ensemble_strategy_task1(
        test_prob_list, strategy_2, task1_ensemble_results[level], task1_model_keys_ordered,
    )

print("Run 1 (submission_1) strategy per level:", best_strategy_per_level)
print("Run 2 (submission_2) strategy per level: uniform soft-vote")

Run 1 (submission_1) strategy per level: {'level1': 'soft_vote_uniform', 'level2': 'weighted_optimized', 'level3': 'weighted_optimized', 'level4': 'hard_vote', 'level5': 'weighted_optimized'}
Run 2 (submission_2) strategy per level: uniform soft-vote


In [ ]:
print("="*70)
print("TASK 1 — RUN 1 (submission_1 strategy) — VALIDATION SET")
print("="*70)

for level in ["level1", "level2", "level3", "level4", "level5"]:
    y_true = task1_val[f"{level}_id"].tolist()
    y_pred = task1_run1_val_preds_raw[level].tolist()
    target_names = list(TASK1_ENCODERS[level].classes_)
    present_labels = sorted(set(y_true) | set(y_pred))
    present_names = [target_names[i] for i in present_labels]

    print(f"\n--- {level} ---")
    print(classification_report(y_true, y_pred, labels=present_labels, target_names=present_names, zero_division=0))

TASK 1 — RUN 1 (submission_1 strategy) — VALIDATION SET

--- level1 ---
              precision    recall  f1-score   support

        Hate       0.81      0.80      0.80       279
    Non-Hate       0.78      0.79      0.79       252

    accuracy                           0.79       531
   macro avg       0.79      0.79      0.79       531
weighted avg       0.80      0.79      0.79       531


--- level2 ---
                     precision    recall  f1-score   support

Hate against LGBTQ+       0.24      0.53      0.33        19
   Hate against Men       0.41      0.79      0.54        78
 Hate against Women       0.52      0.81      0.64       127
                Nil       0.00      0.00      0.00       260
       Profane-Only       0.20      0.60      0.30        47

           accuracy                           0.38       531
          macro avg       0.27      0.55      0.36       531
       weighted avg       0.21      0.38      0.27       531


--- level3 ---
              pre

In [ ]:
print("="*70)
print("TASK 1 — RUN 1 (submission_1 strategy) — TEST SET")
print("="*70)

for level in ["level1", "level2", "level3", "level4", "level5"]:
    y_true = task1_test_labeled[f"{level}_id"].tolist()
    y_pred = task1_run1_test_preds_raw[level].tolist()
    target_names = list(TASK1_ENCODERS[level].classes_)
    present_labels = sorted(set(y_true) | set(y_pred))
    present_names = [target_names[i] for i in present_labels]

    print(f"\n--- {level} ---")
    print(classification_report(y_true, y_pred, labels=present_labels, target_names=present_names, zero_division=0))

TASK 1 — RUN 1 (submission_1 strategy) — TEST SET

--- level1 ---
              precision    recall  f1-score   support

        Hate       0.81      0.79      0.80       279
    Non-Hate       0.77      0.80      0.79       252

    accuracy                           0.79       531
   macro avg       0.79      0.80      0.79       531
weighted avg       0.80      0.79      0.79       531


--- level2 ---
                     precision    recall  f1-score   support

Hate against LGBTQ+       0.15      0.20      0.17        20
   Hate against Men       0.45      0.79      0.57        89
 Hate against Women       0.47      0.84      0.60       120
                Nil       0.00      0.00      0.00       254
       Profane-Only       0.21      0.58      0.31        48

           accuracy                           0.38       531
          macro avg       0.26      0.48      0.33       531
       weighted avg       0.21      0.38      0.27       531


--- level3 ---
              precision

In [ ]:
print("="*70)
print("TASK 1 — RUN 2 (submission_2 strategy: uniform soft-vote) — VALIDATION SET")
print("="*70)

for level in ["level1", "level2", "level3", "level4", "level5"]:
    y_true = task1_val[f"{level}_id"].tolist()
    y_pred = task1_run2_val_preds_raw[level].tolist()
    target_names = list(TASK1_ENCODERS[level].classes_)
    present_labels = sorted(set(y_true) | set(y_pred))
    present_names = [target_names[i] for i in present_labels]

    print(f"\n--- {level} ---")
    print(classification_report(y_true, y_pred, labels=present_labels, target_names=present_names, zero_division=0))

TASK 1 — RUN 2 (submission_2 strategy: uniform soft-vote) — VALIDATION SET

--- level1 ---
              precision    recall  f1-score   support

        Hate       0.81      0.80      0.80       279
    Non-Hate       0.78      0.79      0.79       252

    accuracy                           0.79       531
   macro avg       0.79      0.79      0.79       531
weighted avg       0.80      0.79      0.79       531


--- level2 ---
                     precision    recall  f1-score   support

Hate against LGBTQ+       0.27      0.53      0.36        19
   Hate against Men       0.38      0.74      0.50        78
 Hate against Women       0.51      0.80      0.62       127
                Nil       0.00      0.00      0.00       260
       Profane-Only       0.18      0.53      0.27        47

           accuracy                           0.37       531
          macro avg       0.27      0.52      0.35       531
       weighted avg       0.20      0.37      0.26       531


--- level3 --

In [ ]:
print("="*70)
print("TASK 1 — RUN 2 (submission_2 strategy: uniform soft-vote) — TEST SET")
print("="*70)

for level in ["level1", "level2", "level3", "level4", "level5"]:
    y_true = task1_test_labeled[f"{level}_id"].tolist()
    y_pred = task1_run2_test_preds_raw[level].tolist()
    target_names = list(TASK1_ENCODERS[level].classes_)
    present_labels = sorted(set(y_true) | set(y_pred))
    present_names = [target_names[i] for i in present_labels]

    print(f"\n--- {level} ---")
    print(classification_report(y_true, y_pred, labels=present_labels, target_names=present_names, zero_division=0))

TASK 1 — RUN 2 (submission_2 strategy: uniform soft-vote) — TEST SET

--- level1 ---
              precision    recall  f1-score   support

        Hate       0.81      0.79      0.80       279
    Non-Hate       0.77      0.80      0.79       252

    accuracy                           0.79       531
   macro avg       0.79      0.80      0.79       531
weighted avg       0.80      0.79      0.79       531


--- level2 ---
                     precision    recall  f1-score   support

Hate against LGBTQ+       0.27      0.35      0.30        20
   Hate against Men       0.45      0.83      0.58        89
 Hate against Women       0.47      0.84      0.61       120
                Nil       0.00      0.00      0.00       254
       Profane-Only       0.20      0.52      0.28        48

           accuracy                           0.39       531
          macro avg       0.28      0.51      0.36       531
       weighted avg       0.21      0.39      0.27       531


--- level3 ---
    

In [ ]:
levels_t1 = ["level1", "level2", "level3", "level4", "level5"]


def overall_macro_f1_t1(gold_df, preds_by_level):
    per_level = {}
    for level in levels_t1:
        y_true = gold_df[f"{level}_id"].tolist()
        y_pred = list(preds_by_level[level])
        per_level[level] = f1_score(y_true, y_pred, average="macro", zero_division=0)
    overall = float(np.mean(list(per_level.values())))
    return overall, per_level


def enforce_hierarchy_on_preds(preds_by_level, encoders):
    nonhate_id = int(encoders["level1"].transform(["Non-Hate"])[0])
    l2_nil = int(encoders["level2"].transform(["Nil"])[0])
    l3_nil = int(encoders["level3"].transform(["Nil"])[0])
    l4_nil = int(encoders["level4"].transform(["Nil"])[0])
    l5_nil = int(encoders["level5"].transform(["Nil"])[0])

    out = {}
    out["level1"] = np.array(preds_by_level["level1"])
    l1_nil_mask = out["level1"] == nonhate_id

    l2 = np.where(l1_nil_mask, l2_nil, np.array(preds_by_level["level2"]))
    out["level2"] = l2
    l2_nil_mask = l2 == l2_nil

    l3 = np.where(l2_nil_mask, l3_nil, np.array(preds_by_level["level3"]))
    out["level3"] = l3
    l3_nil_mask = l3 == l3_nil

    out["level4"] = np.where(l3_nil_mask, l4_nil, np.array(preds_by_level["level4"]))
    out["level5"] = np.where(l2_nil_mask, l5_nil, np.array(preds_by_level["level5"]))
    return out


runs_t1 = {
    "run1": {"val": task1_run1_val_preds_raw, "test": task1_run1_test_preds_raw},
    "run2": {"val": task1_run2_val_preds_raw, "test": task1_run2_test_preds_raw},
}

rows = []
per_level_rows = []

for run_name, splits in runs_t1.items():
    for split_name, gold_df in [("val", task1_val), ("test", task1_test_labeled)]:
        raw_preds = splits[split_name]
        raw_overall, raw_per_level = overall_macro_f1_t1(gold_df, raw_preds)

        hier_preds = enforce_hierarchy_on_preds(raw_preds, TASK1_ENCODERS)
        hier_overall, hier_per_level = overall_macro_f1_t1(gold_df, hier_preds)

        rows.append({
            "run": run_name,
            "split": split_name,
            "overall_macro_f1_raw": raw_overall,
            "overall_macro_f1_hierarchy_enforced": hier_overall,
        })

        for level in levels_t1:
            per_level_rows.append({
                "run": run_name,
                "split": split_name,
                "level": level,
                "macro_f1_raw": raw_per_level[level],
                "macro_f1_hierarchy_enforced": hier_per_level[level],
            })

overall_df_t1 = pd.DataFrame(rows)
per_level_df_t1 = pd.DataFrame(per_level_rows)

print("=" * 70)
print("TASK 1 — OVERALL MACRO F1 (mean of level1..level5 macro F1)")
print("=" * 70)
print(overall_df_t1.to_string(index=False))

print("\n" + "=" * 70)
print("TASK 1 — PER-LEVEL MACRO F1 BREAKDOWN")
print("=" * 70)
print(per_level_df_t1.to_string(index=False))

TASK 1 — OVERALL MACRO F1 (mean of level1..level5 macro F1)
 run split  overall_macro_f1_raw  overall_macro_f1_hierarchy_enforced
run1   val              0.371524                             0.479260
run1  test              0.358445                             0.457868
run2   val              0.351633                             0.458549
run2  test              0.355934                             0.458492

TASK 1 — PER-LEVEL MACRO F1 BREAKDOWN
 run split  level  macro_f1_raw  macro_f1_hierarchy_enforced
run1   val level1      0.794374                     0.794374
run1   val level2      0.361080                     0.616542
run1   val level3      0.365570                     0.657788
run1   val level4      0.114464                     0.132909
run1   val level5      0.222132                     0.194689
run1  test level1      0.794491                     0.794491
run1  test level2      0.331064                     0.556776
run1  test level3      0.362972                     0.657928
ru

In [ ]:
summary_rows_t1 = []
for level in ["level1", "level2", "level3", "level4", "level5"]:
    val_true = task1_val[f"{level}_id"].tolist()
    test_true = task1_test_labeled[f"{level}_id"].tolist()

    r1_val_f1 = f1_score(val_true, task1_run1_val_preds_raw[level], average="macro", zero_division=0)
    r1_test_f1 = f1_score(test_true, task1_run1_test_preds_raw[level], average="macro", zero_division=0)
    r2_val_f1 = f1_score(val_true, task1_run2_val_preds_raw[level], average="macro", zero_division=0)
    r2_test_f1 = f1_score(test_true, task1_run2_test_preds_raw[level], average="macro", zero_division=0)

    summary_rows_t1.append({
        "level": level, "run1_val_f1": r1_val_f1, "run1_test_f1": r1_test_f1,
        "run2_val_f1": r2_val_f1, "run2_test_f1": r2_test_f1,
    })

summary_df_t1 = pd.DataFrame(summary_rows_t1)
print("="*70)
print("TASK 1 — RUN 1 vs RUN 2 — VAL vs TEST F1_MACRO SUMMARY")
print("="*70)
print(summary_df_t1.to_string(index=False))

TASK 1 — RUN 1 vs RUN 2 — VAL vs TEST F1_MACRO SUMMARY
 level  run1_val_f1  run1_test_f1  run2_val_f1  run2_test_f1
level1     0.794374      0.794491     0.794374      0.794491
level2     0.361080      0.331064     0.349728      0.356005
level3     0.365570      0.362972     0.358971      0.364229
level4     0.114464      0.086947     0.089897      0.085193
level5     0.222132      0.216750     0.165195      0.179750


In [ ]:
task1_run1_diff_tables = {}

for level in ["level1", "level2", "level3", "level4", "level5"]:
    preds = task1_run1_test_preds_raw[level]
    y_true = task1_test_labeled[f"{level}_id"].values

    val_prob_list = [task1_labeled_test_probs[mk][level] for mk in task1_model_keys_ordered]
    strategy_name = best_strategy_per_level[level]
    if strategy_name == "weighted_optimized":
        weights = task1_ensemble_results[level]["strategies"]["weighted_optimized"]["weights"]
        confidences = np.max(soft_vote_ensemble(val_prob_list, weights), axis=-1)
    elif strategy_name.startswith("single_"):
        mk = strategy_name.replace("single_", "")
        confidences = np.max(task1_labeled_test_probs[mk][level], axis=-1)
    else:
        confidences = np.max(soft_vote_ensemble(val_prob_list), axis=-1)

    diff_df = pd.DataFrame({
        "text": task1_test_labeled["text"].values,
        "actual": TASK1_ENCODERS[level].inverse_transform(y_true),
        "predicted": TASK1_ENCODERS[level].inverse_transform(preds),
        "confidence": confidences,
        "correct": preds == y_true,
    })
    task1_run1_diff_tables[level] = diff_df

    n_total = len(diff_df)
    n_correct = diff_df["correct"].sum()

    print(f"{'='*70}")
    print(f"RUN 1 — {level.upper()} (TEST) — {n_correct}/{n_total} correct ({100*n_correct/n_total:.2f}%)")
    print(f"{'='*70}")
    print("\nAll misclassified rows:")
    mismatches = diff_df[~diff_df["correct"]].sort_values("confidence", ascending=False)
    print(mismatches.to_string())

    print(f"\nError patterns (actual -> predicted):")
    print(mismatches.groupby(["actual", "predicted"]).size().sort_values(ascending=False).head(15))
    print()

RUN 1 — LEVEL1 (TEST) — 422/531 correct (79.47%)

All misclassified rows:
                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                        text    actual predicted  confidence  correct
390                                                                                                                                                                                                                                                                                                                                                                     

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(18, 10))
axes = axes.flatten()

for i, level in enumerate(["level1", "level2", "level3", "level4", "level5"]):
    diff_df = task1_run1_diff_tables[level]
    correct_mask = diff_df["correct"].values
    confidences = diff_df["confidence"].values

    axes[i].hist(confidences[correct_mask], bins=30, alpha=0.6, label="Correct", color="green")
    axes[i].hist(confidences[~correct_mask], bins=30, alpha=0.6, label="Incorrect", color="red")
    axes[i].set_title(f"{level} confidence (Run 1, test)")
    axes[i].set_xlabel("Prediction confidence")
    axes[i].set_ylabel("Count")
    axes[i].legend()

axes[5].axis("off")
plt.suptitle("Task 1 — Run 1 (submission_1) — Test Set Confidence Distribution")
plt.tight_layout()
plt.show()

In [ ]:
task2_test_labeled_raw = pd.read_csv(PATHS.task2_test_labeled)

print("Labeled test shape:", task2_test_labeled_raw.shape)
print("Columns:", list(task2_test_labeled_raw.columns))
print(task2_test_labeled_raw.head(3).to_string())

Labeled test shape: (744, 4)
Columns: ['Id', 'text', 'Coarse-Grained', 'Fine-Grained Hope']
       Id                                                  text     Coarse-Grained Fine-Grained Hope
0  HS_001                  Maatherla movie thule.. super undu..   Encouraging Hope    Inspiring Hope
1  HS_002  ಅಜ್ಜನಾ ಬಾರಿ ಪೋರ್ಲುದ ಸುಗಿಪು... ಮೈ ಮನಸ್ ರೋಮಾಂಚನ ಆಪುಂಡು   Encouraging Hope    Inspiring Hope
2  HS_003                      Maata tuluvere udaldinji Solmelu  Discouraging Hope      Hopelessness


In [ ]:
def validate_and_prepare_task2_labeled_test(df: pd.DataFrame, column_map: Dict[str, str],
                                              encoders: Dict[str, LabelEncoder],
                                              id_column_name: str) -> pd.DataFrame:
    df = df.copy()
    text_col = column_map["text"]

    if text_col not in df.columns:
        raise KeyError(f"Text column '{text_col}' not found. Available: {list(df.columns)}")

    df = preprocess_dataframe_keep_all(df, text_col)
    df = df.rename(columns={text_col: "text"})

    normalize_fns = {"tone": normalize_tone, "hope_type": normalize_hope_type}

    keep_cols = ["text", id_column_name] if id_column_name in df.columns else ["text"]
    for task in ["tone", "hope_type"]:
        col = column_map[task]
        if col not in df.columns:
            raise KeyError(f"Label column '{col}' for {task} not found. Available: {list(df.columns)}")
        df[col] = df[col].astype(str).apply(normalize_fns[task])
        valid_classes = set(encoders[task].classes_)
        invalid_mask = ~df[col].isin(valid_classes)
        if invalid_mask.sum() > 0:
            print(f"WARNING: {invalid_mask.sum()} labeled-test rows in {task} outside label space: "
                  f"{sorted(df.loc[invalid_mask, col].unique())[:10]}")
            df = df[~invalid_mask]
        df[f"{task}_id"] = encoders[task].transform(df[col])
        keep_cols.append(f"{task}_id")

    df = df.reset_index(drop=True)
    return df[keep_cols]


task2_test_labeled = validate_and_prepare_task2_labeled_test(
    task2_test_labeled_raw, TASK2_COLUMN_MAP, TASK2_ENCODERS, id_column_name="Id",
)

print("Labeled test (cleaned) shape:", task2_test_labeled.shape)
for task in ["tone", "hope_type"]:
    print(f"\n{task} distribution (labeled test):")
    print(task2_test_labeled[f"{task}_id"].value_counts().sort_index())

Labeled test (cleaned) shape: (428, 4)

tone distribution (labeled test):
tone_id
0     91
1     94
2    243
Name: count, dtype: int64

hope_type distribution (labeled test):
hope_type_id
0     38
1    126
2    145
3     48
4     71
Name: count, dtype: int64


In [ ]:
task2_labeled_test_probs = {}

for run_key, result in task2_results.items():
    print(f"Collecting labeled-test predictions for {run_key}")
    ckpt_dir = PATHS.output_root / "checkpoints" / "task2"
    tokenizer_path = ckpt_dir / f"{run_key}_tokenizer"
    checkpoint_path = Path(result["checkpoint_path"])

    probs = get_task2_predictions(
        run_key, result["model_name"], checkpoint_path, tokenizer_path,
        task2_test_labeled, task2_config.max_seq_len, task2_config.eval_batch_size, has_labels=False,
    )
    task2_labeled_test_probs[run_key] = probs
    clear_memory()

print("Collected labeled-test probabilities for:", list(task2_labeled_test_probs.keys()))

Collected labeled-test probabilities for: ['xlm-roberta-base_equal', 'mbert-base_equal', 'xlm-roberta-base_tone_heavy', 'xlm-roberta-base_hope_type_heavy', 'xlm-roberta-base_balanced_070_030']


In [ ]:
def apply_ensemble_strategy_task2(prob_list: List[np.ndarray], strategy_name: str,
                                   ensemble_results_for_task: Dict[str, Any],
                                   run_keys_ordered: List[str]) -> np.ndarray:
    if strategy_name == "hard_vote":
        return hard_vote_ensemble(prob_list)
    elif strategy_name == "soft_vote_uniform":
        return np.argmax(soft_vote_ensemble(prob_list), axis=-1)
    elif strategy_name == "rank_average":
        return np.argmax(rank_average_ensemble(prob_list), axis=-1)
    elif strategy_name == "logit_average":
        return np.argmax(logit_average_ensemble(prob_list), axis=-1)
    elif strategy_name == "weighted_optimized":
        weights = ensemble_results_for_task["strategies"]["weighted_optimized"]["weights"]
        return np.argmax(soft_vote_ensemble(prob_list, weights), axis=-1)
    elif strategy_name.startswith("single_"):
        rk = strategy_name.replace("single_", "")
        idx = run_keys_ordered.index(rk)
        return np.argmax(prob_list[idx], axis=-1)
    else:
        return np.argmax(soft_vote_ensemble(prob_list), axis=-1)


task2_run1_val_preds_raw = {}
task2_run2_val_preds_raw = {}
task2_run1_test_preds_raw = {}
task2_run2_test_preds_raw = {}

for task in ["tone", "hope_type"]:
    val_prob_list = [task2_val_probs[rk][task] for rk in cross_model_keys]
    test_prob_list = [task2_labeled_test_probs[rk][task] for rk in cross_model_keys]

    strategy_1 = best_strategy_per_task_2[task]
    strategy_2 = "soft_vote_uniform"

    task2_run1_val_preds_raw[task] = apply_ensemble_strategy_task2(
        val_prob_list, strategy_1, task2_ensemble_results[task], cross_model_keys,
    )
    task2_run2_val_preds_raw[task] = apply_ensemble_strategy_task2(
        val_prob_list, strategy_2, task2_ensemble_results[task], cross_model_keys,
    )
    task2_run1_test_preds_raw[task] = apply_ensemble_strategy_task2(
        test_prob_list, strategy_1, task2_ensemble_results[task], cross_model_keys,
    )
    task2_run2_test_preds_raw[task] = apply_ensemble_strategy_task2(
        test_prob_list, strategy_2, task2_ensemble_results[task], cross_model_keys,
    )

print("Run 1 (submission_1) strategy per task:", best_strategy_per_task_2)
print("Run 2 (submission_2) strategy per task: uniform soft-vote")

Run 1 (submission_1) strategy per task: {'tone': 'weighted_optimized', 'hope_type': 'weighted_optimized'}
Run 2 (submission_2) strategy per task: uniform soft-vote


In [ ]:
print("="*70)
print("TASK 2 — RUN 1 (submission_1 strategy) — VALIDATION SET")
print("="*70)

for task in ["tone", "hope_type"]:
    y_true = task2_val[f"{task}_id"].tolist()
    y_pred = task2_run1_val_preds_raw[task].tolist()
    target_names = list(TASK2_ENCODERS[task].classes_)
    present_labels = sorted(set(y_true) | set(y_pred))
    present_names = [target_names[i] for i in present_labels]

    print(f"\n--- {task} ---")
    print(classification_report(y_true, y_pred, labels=present_labels, target_names=present_names, zero_division=0))

TASK 2 — RUN 1 (submission_1 strategy) — VALIDATION SET

--- tone ---
              precision    recall  f1-score   support

     Blended       0.41      0.36      0.38       163
Discouraging       0.37      0.44      0.40       164
 Encouraging       0.79      0.76      0.78       426

    accuracy                           0.61       753
   macro avg       0.52      0.52      0.52       753
weighted avg       0.62      0.61      0.61       753


--- hope_type ---
                 precision    recall  f1-score   support

    Fading Hope       0.27      0.28      0.28        57
   Hopelessness       0.55      0.55      0.55       213
 Inspiring Hope       0.60      0.72      0.66       262
Optimistic Hope       0.35      0.56      0.43        98
 Realistic Hope       0.11      0.01      0.02       123

       accuracy                           0.50       753
      macro avg       0.38      0.43      0.39       753
   weighted avg       0.45      0.50      0.46       753



In [ ]:
print("="*70)
print("TASK 2 — RUN 1 (submission_1 strategy) — TEST SET")
print("="*70)

for task in ["tone", "hope_type"]:
    y_true = task2_test_labeled[f"{task}_id"].tolist()
    y_pred = task2_run1_test_preds_raw[task].tolist()
    target_names = list(TASK2_ENCODERS[task].classes_)
    present_labels = sorted(set(y_true) | set(y_pred))
    present_names = [target_names[i] for i in present_labels]

    print(f"\n--- {task} ---")
    print(classification_report(y_true, y_pred, labels=present_labels, target_names=present_names, zero_division=0))

TASK 2 — RUN 1 (submission_1 strategy) — TEST SET

--- tone ---
              precision    recall  f1-score   support

     Blended       0.37      0.32      0.34        91
Discouraging       0.27      0.28      0.28        94
 Encouraging       0.66      0.70      0.68       243

    accuracy                           0.52       428
   macro avg       0.44      0.43      0.43       428
weighted avg       0.52      0.52      0.52       428


--- hope_type ---
                 precision    recall  f1-score   support

    Fading Hope       0.26      0.26      0.26        38
   Hopelessness       0.39      0.33      0.35       126
 Inspiring Hope       0.46      0.60      0.52       145
Optimistic Hope       0.17      0.29      0.22        48
 Realistic Hope       0.45      0.07      0.12        71

       accuracy                           0.37       428
      macro avg       0.35      0.31      0.29       428
   weighted avg       0.39      0.37      0.35       428



In [ ]:
print("="*70)
print("TASK 2 — RUN 2 (submission_2 strategy: uniform soft-vote) — VALIDATION SET")
print("="*70)

for task in ["tone", "hope_type"]:
    y_true = task2_val[f"{task}_id"].tolist()
    y_pred = task2_run2_val_preds_raw[task].tolist()
    target_names = list(TASK2_ENCODERS[task].classes_)
    present_labels = sorted(set(y_true) | set(y_pred))
    present_names = [target_names[i] for i in present_labels]

    print(f"\n--- {task} ---")
    print(classification_report(y_true, y_pred, labels=present_labels, target_names=present_names, zero_division=0))

TASK 2 — RUN 2 (submission_2 strategy: uniform soft-vote) — VALIDATION SET

--- tone ---
              precision    recall  f1-score   support

     Blended       0.41      0.40      0.40       163
Discouraging       0.33      0.29      0.31       164
 Encouraging       0.75      0.79      0.77       426

    accuracy                           0.60       753
   macro avg       0.50      0.49      0.49       753
weighted avg       0.59      0.60      0.59       753


--- hope_type ---
                 precision    recall  f1-score   support

    Fading Hope       0.23      0.19      0.21        57
   Hopelessness       0.53      0.53      0.53       213
 Inspiring Hope       0.57      0.79      0.67       262
Optimistic Hope       0.34      0.45      0.39        98
 Realistic Hope       0.00      0.00      0.00       123

       accuracy                           0.50       753
      macro avg       0.33      0.39      0.36       753
   weighted avg       0.41      0.50      0.45       

In [ ]:
print("="*70)
print("TASK 2 — RUN 2 (submission_2 strategy: uniform soft-vote) — TEST SET")
print("="*70)

for task in ["tone", "hope_type"]:
    y_true = task2_test_labeled[f"{task}_id"].tolist()
    y_pred = task2_run2_test_preds_raw[task].tolist()
    target_names = list(TASK2_ENCODERS[task].classes_)
    present_labels = sorted(set(y_true) | set(y_pred))
    present_names = [target_names[i] for i in present_labels]

    print(f"\n--- {task} ---")
    print(classification_report(y_true, y_pred, labels=present_labels, target_names=present_names, zero_division=0))

TASK 2 — RUN 2 (submission_2 strategy: uniform soft-vote) — TEST SET

--- tone ---
              precision    recall  f1-score   support

     Blended       0.32      0.31      0.31        91
Discouraging       0.23      0.16      0.19        94
 Encouraging       0.65      0.74      0.69       243

    accuracy                           0.52       428
   macro avg       0.40      0.40      0.40       428
weighted avg       0.49      0.52      0.50       428


--- hope_type ---
                 precision    recall  f1-score   support

    Fading Hope       0.26      0.24      0.25        38
   Hopelessness       0.41      0.33      0.36       126
 Inspiring Hope       0.42      0.66      0.52       145
Optimistic Hope       0.15      0.19      0.17        48
 Realistic Hope       0.25      0.03      0.05        71

       accuracy                           0.37       428
      macro avg       0.30      0.29      0.27       428
   weighted avg       0.35      0.37      0.33       428



In [ ]:
summary_rows_t2 = []
for task in ["tone", "hope_type"]:
    val_true = task2_val[f"{task}_id"].tolist()
    test_true = task2_test_labeled[f"{task}_id"].tolist()

    r1_val_f1 = f1_score(val_true, task2_run1_val_preds_raw[task], average="macro", zero_division=0)
    r1_test_f1 = f1_score(test_true, task2_run1_test_preds_raw[task], average="macro", zero_division=0)
    r2_val_f1 = f1_score(val_true, task2_run2_val_preds_raw[task], average="macro", zero_division=0)
    r2_test_f1 = f1_score(test_true, task2_run2_test_preds_raw[task], average="macro", zero_division=0)

    summary_rows_t2.append({
        "task": task, "run1_val_f1": r1_val_f1, "run1_test_f1": r1_test_f1,
        "run2_val_f1": r2_val_f1, "run2_test_f1": r2_test_f1,
    })

summary_df_t2 = pd.DataFrame(summary_rows_t2)
print("="*70)
print("TASK 2 — RUN 1 vs RUN 2 — VAL vs TEST F1_MACRO SUMMARY")
print("="*70)
print(summary_df_t2.to_string(index=False))

TASK 2 — RUN 1 vs RUN 2 — VAL vs TEST F1_MACRO SUMMARY
     task  run1_val_f1  run1_test_f1  run2_val_f1  run2_test_f1
     tone     0.519531      0.432347     0.494817      0.397742
hope_type     0.386378      0.294360     0.357969      0.269201


In [ ]:
task2_run1_diff_tables = {}

for task in ["tone", "hope_type"]:
    preds = task2_run1_test_preds_raw[task]
    y_true = task2_test_labeled[f"{task}_id"].values

    test_prob_list = [task2_labeled_test_probs[rk][task] for rk in cross_model_keys]
    strategy_name = best_strategy_per_task_2[task]
    if strategy_name == "weighted_optimized":
        weights = task2_ensemble_results[task]["strategies"]["weighted_optimized"]["weights"]
        confidences = np.max(soft_vote_ensemble(test_prob_list, weights), axis=-1)
    elif strategy_name.startswith("single_"):
        rk = strategy_name.replace("single_", "")
        confidences = np.max(task2_labeled_test_probs[rk][task], axis=-1)
    else:
        confidences = np.max(soft_vote_ensemble(test_prob_list), axis=-1)

    diff_df = pd.DataFrame({
        "text": task2_test_labeled["text"].values,
        "actual": TASK2_ENCODERS[task].inverse_transform(y_true),
        "predicted": TASK2_ENCODERS[task].inverse_transform(preds),
        "confidence": confidences,
        "correct": preds == y_true,
    })
    task2_run1_diff_tables[task] = diff_df

    n_total = len(diff_df)
    n_correct = diff_df["correct"].sum()

    print(f"{'='*70}")
    print(f"RUN 1 — {task.upper()} (TEST) — {n_correct}/{n_total} correct ({100*n_correct/n_total:.2f}%)")
    print(f"{'='*70}")
    print("\nAll misclassified rows:")
    mismatches = diff_df[~diff_df["correct"]].sort_values("confidence", ascending=False)
    print(mismatches.to_string())

    print(f"\nError patterns (actual -> predicted):")
    print(mismatches.groupby(["actual", "predicted"]).size().sort_values(ascending=False).head(15))
    print()

RUN 1 — TONE (TEST) — 224/428 correct (52.34%)

All misclassified rows:
                                                                                                       text        actual     predicted  confidence  correct
296                                                           dheeraj sir spr bhagavathi bhari edde athndye  Discouraging   Encouraging    0.839499    False
367                                                                    padya super i love you arjun kapikad  Discouraging   Encouraging    0.838721    False
315                                                porlu athund nataka thudh masth kushi aand talkies app d  Discouraging   Encouraging    0.836021    False
400                                                                   mast eddeye athnd baari bega mugithar  Discouraging   Encouraging    0.835893    False
313                                                         very nice yaxaganadakulu saiter, i like kapikad       Blended   Encouraging    0.83

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for i, task in enumerate(["tone", "hope_type"]):
    diff_df = task2_run1_diff_tables[task]
    correct_mask = diff_df["correct"].values
    confidences = diff_df["confidence"].values

    axes[i].hist(confidences[correct_mask], bins=25, alpha=0.6, label="Correct", color="green")
    axes[i].hist(confidences[~correct_mask], bins=25, alpha=0.6, label="Incorrect", color="red")
    axes[i].set_title(f"{task} confidence (Run 1, test)")
    axes[i].set_xlabel("Prediction confidence")
    axes[i].set_ylabel("Count")
    axes[i].legend()

plt.suptitle("Task 2 — Run 1 (submission_1) — Test Set Confidence Distribution")
plt.tight_layout()
plt.show()

In [ ]:
task3_test_labeled_raw = pd.read_csv(PATHS.task3_test_labeled)

print("Labeled test shape:", task3_test_labeled_raw.shape)
print("Columns:", list(task3_test_labeled_raw.columns))
print(task3_test_labeled_raw.head(3).to_string())

Labeled test shape: (500, 4)
Columns: ['Unnamed: 0', 'ID', 'Comments', 'Labels']
   Unnamed: 0      ID                                                             Comments  Labels
0           0  TEST_0                                      ನಿನಗೆ ಪ್ರಶ್ನೆ ಕೇಳೋಕೆ ಬರ್ತಿಲ್ಲಾ.       0
1           1  TEST_1  Thika muchkond hogo bolinagane... Avn tunne unnu neen uddara agtiya       1
2           2  TEST_2                                    Abdul razak he is NO 1 terrorist.       1


In [ ]:
def validate_and_prepare_task3_labeled_test(df: pd.DataFrame, column_map: Dict[str, str],
                                              raw_label_map: Dict[str, str],
                                              encoders: Dict[str, LabelEncoder],
                                              id_column_name: str) -> pd.DataFrame:
    df = df.copy()
    text_col = column_map["text"]

    if text_col not in df.columns:
        raise KeyError(f"Text column '{text_col}' not found. Available: {list(df.columns)}")

    df = preprocess_dataframe_keep_all(df, text_col, lowercase_latin_only=True, remove_emoji=False)
    df = df.rename(columns={text_col: "text"})

    label_col = column_map["label"]
    if label_col not in df.columns:
        raise KeyError(f"Label column '{label_col}' not found. Available: {list(df.columns)}")

    df[label_col] = df[label_col].astype(str).str.strip()
    df[label_col] = df[label_col].map(raw_label_map)

    invalid_mask = df[label_col].isna()
    if invalid_mask.sum() > 0:
        print(f"WARNING: {invalid_mask.sum()} labeled-test rows had unmapped raw label values and were dropped")
        df = df[~invalid_mask]

    valid_classes = set(encoders["label"].classes_)
    invalid_mask2 = ~df[label_col].isin(valid_classes)
    if invalid_mask2.sum() > 0:
        df = df[~invalid_mask2]

    df["label_id"] = encoders["label"].transform(df[label_col])
    df = df.reset_index(drop=True)

    keep_cols = ["text", "label_id"]
    if id_column_name in df.columns:
        keep_cols = ["text", id_column_name, "label_id"]

    return df[keep_cols]


task3_test_labeled = validate_and_prepare_task3_labeled_test(
    task3_test_labeled_raw, TASK3_COLUMN_MAP, TASK3_RAW_LABEL_MAP, TASK3_ENCODERS, id_column_name="ID",
)

print("Labeled test (cleaned) shape:", task3_test_labeled.shape)
print("\nLabel distribution (labeled test):")
print(task3_test_labeled["label_id"].value_counts().sort_index())

Labeled test (cleaned) shape: (500, 3)

Label distribution (labeled test):
label_id
0    225
1    275
Name: count, dtype: int64


In [ ]:
task3_labeled_test_probs = {}

for model_key, result in task3_results.items():
    print(f"Collecting labeled-test predictions for {model_key}")
    ckpt_dir = PATHS.output_root / "checkpoints" / "task3"
    tokenizer_path = ckpt_dir / f"{model_key}_tokenizer"
    checkpoint_path = Path(result["checkpoint_path"])

    probs = get_task3_predictions(
        model_key, result["model_name"], checkpoint_path, tokenizer_path,
        task3_test_labeled, task3_config.max_seq_len, task3_config.eval_batch_size, has_labels=False,
    )
    task3_labeled_test_probs[model_key] = probs
    clear_memory()

print("Collected labeled-test probabilities for:", list(task3_labeled_test_probs.keys()))

Collected labeled-test probabilities for: ['xlm-roberta-base', 'mbert-base']


In [ ]:
def apply_ensemble_strategy_task3(prob_list: List[np.ndarray], strategy_name: str,
                                   ensemble_results: Dict[str, Any], hate_class_id: int,
                                   model_keys_ordered: List[str]) -> np.ndarray:
    strategy_info = ensemble_results["strategies"].get(strategy_name, {})

    if strategy_name == "hard_vote":
        return hard_vote_ensemble(prob_list)
    elif strategy_name == "soft_vote_uniform":
        return np.argmax(soft_vote_ensemble(prob_list), axis=-1)
    elif strategy_name == "rank_average":
        return np.argmax(rank_average_ensemble(prob_list), axis=-1)
    elif strategy_name == "logit_average":
        return np.argmax(logit_average_ensemble(prob_list), axis=-1)
    elif strategy_name == "weighted_optimized":
        weights = strategy_info["weights"]
        return np.argmax(soft_vote_ensemble(prob_list, weights), axis=-1)
    elif strategy_name == "soft_vote_thresholded":
        threshold = strategy_info["threshold"]
        soft_probs = soft_vote_ensemble(prob_list)
        hate_probs = soft_probs[:, hate_class_id]
        return np.array([hate_class_id if p >= threshold else (1 - hate_class_id) for p in hate_probs])
    elif strategy_name.startswith("single_") and strategy_name.endswith("_thresholded"):
        mk = strategy_name.replace("single_", "").replace("_thresholded", "")
        idx = model_keys_ordered.index(mk)
        threshold = strategy_info["threshold"]
        hate_probs = prob_list[idx][:, hate_class_id]
        return np.array([hate_class_id if p >= threshold else (1 - hate_class_id) for p in hate_probs])
    elif strategy_name.startswith("single_"):
        mk = strategy_name.replace("single_", "")
        idx = model_keys_ordered.index(mk)
        return np.argmax(prob_list[idx], axis=-1)
    else:
        return np.argmax(soft_vote_ensemble(prob_list), axis=-1)


task3_val_prob_list_ordered = [np.array(task3_results[mk]["val_probs"]) for mk in task3_model_keys]
task3_test_prob_list_ordered = [task3_labeled_test_probs[mk] for mk in task3_model_keys]

task3_run1_val_preds_raw = apply_ensemble_strategy_task3(
    task3_val_prob_list_ordered, submission_1_strategy_t3, task3_ensemble_results, TASK3_HATE_ID, task3_model_keys,
)
task3_run2_val_preds_raw = apply_ensemble_strategy_task3(
    task3_val_prob_list_ordered, submission_2_strategy_t3, task3_ensemble_results, TASK3_HATE_ID, task3_model_keys,
)
task3_run1_test_preds_raw = apply_ensemble_strategy_task3(
    task3_test_prob_list_ordered, submission_1_strategy_t3, task3_ensemble_results, TASK3_HATE_ID, task3_model_keys,
)
task3_run2_test_preds_raw = apply_ensemble_strategy_task3(
    task3_test_prob_list_ordered, submission_2_strategy_t3, task3_ensemble_results, TASK3_HATE_ID, task3_model_keys,
)

print(f"Run 1 (submission_1) strategy: {submission_1_strategy_t3}")
print(f"Run 2 (submission_2) strategy: {submission_2_strategy_t3}")

Run 1 (submission_1) strategy: soft_vote_thresholded
Run 2 (submission_2) strategy: soft_vote_thresholded


In [ ]:
print("="*70)
print(f"TASK 3 — RUN 1 ({submission_1_strategy_t3}) — VALIDATION SET")
print("="*70)
print(classification_report(
    task3_y_true_val, task3_run1_val_preds_raw, target_names=list(TASK3_ENCODERS["label"].classes_), zero_division=0,
))

TASK 3 — RUN 1 (soft_vote_thresholded) — VALIDATION SET
              precision    recall  f1-score   support

        Hate       0.70      0.79      0.74       212
    Not-hate       0.83      0.75      0.79       288

    accuracy                           0.77       500
   macro avg       0.76      0.77      0.77       500
weighted avg       0.77      0.77      0.77       500



In [ ]:
task3_y_true_test = task3_test_labeled["label_id"].tolist()

print("="*70)
print(f"TASK 3 — RUN 1 ({submission_1_strategy_t3}) — TEST SET")
print("="*70)
print(classification_report(
    task3_y_true_test, task3_run1_test_preds_raw, target_names=list(TASK3_ENCODERS["label"].classes_), zero_division=0,
))

TASK 3 — RUN 1 (soft_vote_thresholded) — TEST SET
              precision    recall  f1-score   support

        Hate       0.70      0.86      0.77       225
    Not-hate       0.86      0.69      0.77       275

    accuracy                           0.77       500
   macro avg       0.78      0.78      0.77       500
weighted avg       0.78      0.77      0.77       500



In [ ]:
print("="*70)
print(f"TASK 3 — RUN 2 ({submission_2_strategy_t3}) — VALIDATION SET")
print("="*70)
print(classification_report(
    task3_y_true_val, task3_run2_val_preds_raw, target_names=list(TASK3_ENCODERS["label"].classes_), zero_division=0,
))

TASK 3 — RUN 2 (soft_vote_thresholded) — VALIDATION SET
              precision    recall  f1-score   support

        Hate       0.70      0.79      0.74       212
    Not-hate       0.83      0.75      0.79       288

    accuracy                           0.77       500
   macro avg       0.76      0.77      0.77       500
weighted avg       0.77      0.77      0.77       500



In [ ]:
print("="*70)
print(f"TASK 3 — RUN 2 ({submission_2_strategy_t3}) — TEST SET")
print("="*70)
print(classification_report(
    task3_y_true_test, task3_run2_test_preds_raw, target_names=list(TASK3_ENCODERS["label"].classes_), zero_division=0,
))

TASK 3 — RUN 2 (soft_vote_thresholded) — TEST SET
              precision    recall  f1-score   support

        Hate       0.70      0.86      0.77       225
    Not-hate       0.86      0.69      0.77       275

    accuracy                           0.77       500
   macro avg       0.78      0.78      0.77       500
weighted avg       0.78      0.77      0.77       500



In [ ]:
task3_run3_val_preds_raw = apply_ensemble_strategy_task3(
    task3_val_prob_list_ordered, submission_3_strategy_t3, task3_ensemble_results, TASK3_HATE_ID, task3_model_keys,
)
task3_run3_test_preds_raw = apply_ensemble_strategy_task3(
    task3_test_prob_list_ordered, submission_3_strategy_t3, task3_ensemble_results, TASK3_HATE_ID, task3_model_keys,
)

print(f"Run 3 (submission_3) strategy: {submission_3_strategy_t3}")
print(f"Best single model: {best_single_model_t3}")
print(f"Threshold used: {task3_thresholds.get(best_single_model_t3, 0.5):.2f}")

Run 3 (submission_3) strategy: single_mbert-base_thresholded
Best single model: mbert-base
Threshold used: 0.40


In [ ]:
print("="*70)
print(f"TASK 3 — RUN 3 ({submission_3_strategy_t3}) — VALIDATION SET")
print("="*70)
print(classification_report(
    task3_y_true_val, task3_run3_val_preds_raw, target_names=list(TASK3_ENCODERS["label"].classes_), zero_division=0,
))

TASK 3 — RUN 3 (single_mbert-base_thresholded) — VALIDATION SET
              precision    recall  f1-score   support

        Hate       0.67      0.80      0.73       212
    Not-hate       0.83      0.71      0.76       288

    accuracy                           0.75       500
   macro avg       0.75      0.75      0.74       500
weighted avg       0.76      0.75      0.75       500



In [ ]:
print("="*70)
print(f"TASK 3 — RUN 3 ({submission_3_strategy_t3}) — TEST SET")
print("="*70)
print(classification_report(
    task3_y_true_test, task3_run3_test_preds_raw, target_names=list(TASK3_ENCODERS["label"].classes_), zero_division=0,
))

TASK 3 — RUN 3 (single_mbert-base_thresholded) — TEST SET
              precision    recall  f1-score   support

        Hate       0.67      0.82      0.74       225
    Not-hate       0.82      0.67      0.73       275

    accuracy                           0.74       500
   macro avg       0.74      0.74      0.74       500
weighted avg       0.75      0.74      0.74       500



In [ ]:
r1_val_f1_t3 = f1_score(task3_y_true_val, task3_run1_val_preds_raw, average="macro", zero_division=0)
r1_test_f1_t3 = f1_score(task3_y_true_test, task3_run1_test_preds_raw, average="macro", zero_division=0)
r2_val_f1_t3 = f1_score(task3_y_true_val, task3_run2_val_preds_raw, average="macro", zero_division=0)
r2_test_f1_t3 = f1_score(task3_y_true_test, task3_run2_test_preds_raw, average="macro", zero_division=0)

summary_df_t3 = pd.DataFrame([{
    "run1_val_f1": r1_val_f1_t3, "run1_test_f1": r1_test_f1_t3,
    "run2_val_f1": r2_val_f1_t3, "run2_test_f1": r2_test_f1_t3,
}])
print("="*70)
print("TASK 3 — RUN 1 vs RUN 2 — VAL vs TEST F1_MACRO SUMMARY")
print("="*70)
print(summary_df_t3.to_string(index=False))

TASK 3 — RUN 1 vs RUN 2 — VAL vs TEST F1_MACRO SUMMARY
 run1_val_f1  run1_test_f1  run2_val_f1  run2_test_f1
    0.765657      0.767996     0.765657      0.767996


In [ ]:
strategy_info_t3 = task3_ensemble_results["strategies"].get(submission_1_strategy_t3, {})

if submission_1_strategy_t3 == "soft_vote_thresholded":
    confidences_t3 = np.max(soft_vote_ensemble(task3_test_prob_list_ordered), axis=-1)
elif submission_1_strategy_t3.startswith("single_"):
    mk = submission_1_strategy_t3.replace("single_", "").replace("_thresholded", "")
    confidences_t3 = np.max(task3_labeled_test_probs[mk], axis=-1)
else:
    confidences_t3 = np.max(soft_vote_ensemble(task3_test_prob_list_ordered), axis=-1)

diff_df_t3_run1 = pd.DataFrame({
    "text": task3_test_labeled["text"].values,
    "actual": TASK3_ENCODERS["label"].inverse_transform(task3_y_true_test),
    "predicted": TASK3_ENCODERS["label"].inverse_transform(task3_run1_test_preds_raw),
    "confidence": confidences_t3,
    "correct": task3_run1_test_preds_raw == np.array(task3_y_true_test),
})

n_total = len(diff_df_t3_run1)
n_correct = diff_df_t3_run1["correct"].sum()

print(f"{'='*70}")
print(f"RUN 1 — TASK 3 (TEST) — {n_correct}/{n_total} correct ({100*n_correct/n_total:.2f}%)")
print(f"{'='*70}")
print("\nAll misclassified rows:")
mismatches_t3 = diff_df_t3_run1[~diff_df_t3_run1["correct"]].sort_values("confidence", ascending=False)
print(mismatches_t3.to_string())

print(f"\nError patterns (actual -> predicted):")
print(mismatches_t3.groupby(["actual", "predicted"]).size().sort_values(ascending=False))

RUN 1 — TASK 3 (TEST) — 384/500 correct (76.80%)

All misclassified rows:
                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                         text    actual predicted  confidence  correct
296                                                                    

In [ ]:
confidences_t3_run3 = np.max(task3_labeled_test_probs[best_single_model_t3], axis=-1)

diff_df_t3_run3 = pd.DataFrame({
    "text": task3_test_labeled["text"].values,
    "actual": TASK3_ENCODERS["label"].inverse_transform(task3_y_true_test),
    "predicted": TASK3_ENCODERS["label"].inverse_transform(task3_run3_test_preds_raw),
    "confidence": confidences_t3_run3,
    "correct": task3_run3_test_preds_raw == np.array(task3_y_true_test),
})

n_total = len(diff_df_t3_run3)
n_correct = diff_df_t3_run3["correct"].sum()

print(f"{'='*70}")
print(f"RUN 3 — TASK 3 (TEST) — {n_correct}/{n_total} correct ({100*n_correct/n_total:.2f}%)")
print(f"{'='*70}")
print("\nAll misclassified rows:")
mismatches_t3_run3 = diff_df_t3_run3[~diff_df_t3_run3["correct"]].sort_values("confidence", ascending=False)
print(mismatches_t3_run3.to_string())

print(f"\nError patterns (actual -> predicted):")
print(mismatches_t3_run3.groupby(["actual", "predicted"]).size().sort_values(ascending=False))

RUN 3 — TASK 3 (TEST) — 368/500 correct (73.60%)

All misclassified rows:
                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                              

: 